# Deney 05 — Viral Potansiyel / İzlenme Tahmini (gerçek geçmiş veriyle)

**Amaç:** Notebook 04'ün ürettiği gerçek `youtube_data/normalized_videos.csv` geçmiş veri setiyle istatistiksel baseline'lar kurmak, opsiyonel olarak LLM tahminini bu baseline'larla karşılaştırmak. Yeterli veri yoksa (`MIN_HISTORY_ROWS` altında) notebook **kendinden emin bir tahmin üretmez** — `status='insufficient_data'` ile açıkça durur.

**Çalıştırma sırası:** Notebook 04 (gerçek geçmiş veri toplama) → **bu notebook**.

API anahtarları (Colab 🔑 Secrets, opsiyonel): `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. LLM tahmini sadece yeterli gerçek veri VE en az bir anahtar varsa çalışır; asla birkaç elle girilmiş satırdan güven skoru uydurmaz.


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **viral sonuc yorumlama** (KRITIK OLMAYAN) -> Rota B (`gemini-3.5-flash-lite` -> `gemini-3.1-flash-lite` -> `gemma-4-26b-a4b-it`). Istatistiksel baseline'in YERINE GECMEZ, onunla karsilastirilir.


In [ ]:
!pip -q install openai anthropic google-genai pandas numpy > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, time, json, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)
print('OpenAI:', 'var' if OPENAI_API_KEY else 'yok')
print('Anthropic:', 'var' if ANTHROPIC_API_KEY else 'yok')
print('Google:', 'var' if GOOGLE_API_KEY else 'yok')

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []
def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({'task': task, 'candidate': candidate, 'quality_score_1_5': quality_score,
                     'cost_usd_estimate': round(cost_usd, 5), 'latency_sec': round(latency_sec, 2),
                     'notes': notes, 'status': status})
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} status={status}")
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/AVRUPA YAKASI/YoutubeOtomationv0"


## Kalıcı depolama ve `local_pipeline.youtube_data` modülü

Notebook 04 ile aynı modül (`local_pipeline/youtube_data.py`) burada da bootstrap hücresiyle yüklenir; kopya kod yoktur.

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['youtube_data', 'viral_prediction', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'youtube_data': {
        'sha256': '048809136491091e9650d1783a6708cf1bc59c4a719983c19ccb1fb85b4a9f69',
        'encoding': 'zlib+b85',
        'data': (
            'c-q~4*>>Ybvfw+vB8T_(6;P%qmF@EMuuS`^CH1w&wRK75ZnvyPfkaY53k29$q++S{5p&MG&o9m=Of0zqpr~@YUdHFRL;$%)M#h#A'
            'k;CEet!%1zmdTMQWt{95dA1fm7tL8Slj10@V{!22MBIEO=0%ptd7T!y$coE!J{}BCui{cBVzw6hUq^K*^CT)(tD>x%JgwJJH809E'
            'zZ|Elbv_%3{b$ilTE<zlf=Vg1q(8-ZGT7gb)<si8w^7{G#WKbM(K4=TS#mjXlg1(zv!bZ0x{Oz1F0)LexsdVvO5{Z?;nTGUDk%r='
            'vto{2r7M}Gx!luDo61p@r>m8$$E)>wu_#JWUr8Z9#Pizepq3x%L46h1BCUj6&Sa9neAB!xpm<S9(`glost_>@r@kts6!Vxy2IXGA'
            'Iv%9URVJ4*uW3&39_AD$qFCtC#VVDRppNE6lO-Z4(?u;{%vh*e=LBszZ%TkG6v<(9Fwh_=a0CFCB8zJ&Zm%Q&BS~w3I<y93UZj}}'
            'Cj%jN#Th;?i=@e5*z!h}wB8i}w372u)?!f>OK}J-%|uhlGQqV6xD?e`=CE)ua2$JGL2E)3w>S^z38r)-pVNe8S&3C#RRl`JnrpeH'
            '<@vD6(s^0~VgW$j&zdw#qIs4AK;DmOY_AF!EsXg}maqsowiRGbU5yac;9nTV3R+LAnj0_UTipRS@V<(1Fdt<k%d#jVT)6jQ&w1aJ'
            '*?aNw-RYSCypC5eB3MZnPMoh{fOEORrJ<=|UEB=Ss>or9XGOB65xjf*0_p;)NPq$&#4W5w4G_mx#Q!{f{c7*<^d|y26sM+W7BU$@'
            'y?G2k6_-VEnaOyS^7L)b6+jRnI?tL!(i|5Jpin9Q(a5T%#<2h3_9_KjM+}Goji?|(ImUH_qU9P^pp2IiHcdG0tULTNn|y+^U_`*h'
            'g9c!w@?=%OkX=-Ss_9}qq9HV^tcVj{k7kyo)fK{z*A9?Fk&W?SI2;ZLouX&~`>vExBw!N&Wq|U4v9u4W!9cycimR(Eo$1HFSFjT5'
            'H;{oV{ky32uTtt?b-I+?72vDHC*6^L9#NS@W_8RzKNh*FvVz8;UtQx3_BA*?J2-oH8vS(e;@$D7n2I}^`4Es;mMLxGY&a6b1R$3c'
            'D|`{HvUo19iY$>Oek$cnDsLlPyA(*>kYP6jq$`UXd=^7kyJoR~O@Y;Mr{T**45&k&p<vPwg^X~$*3iWLVDReo+m{C~PX2m)6#aB^'
            'bo@FxeEs6x%U3wabME3MO=J<J)J6>(PVz{rd6^QHQQN$5GKu0EpXXPwBr;RA^pmQsn}z8BHsn+xORuGUniorgfq9WMC6n&7Dt`aX'
            '{!{NAkQ&`+9bZ<|0IbtxQLfbh61j+*td6oczii@5$wg@NQG5dsLSCf`5&^<laf`6qZsIxc9l)=K+D@yex`L&R%m84QB`Kl?kDXYW'
            '0>$8BFgQ9nJ$(Js@!K8{Z-o9A8v1~%bA&7Gy(M+8U(m+15UONc&aa}P1bm@#ylgPp=&ES&l?8hh0ou`PEGY9!C`Irt)f^BD=bGEf'
            'Rw4C{n=9p9=5wc@Uh7h>WL(?#%*oATlETj$oIx4q*Bv+={&4W>)$xliGCqh{7C1b41DqBR6|Mlu$RZyNm+^<F3!$Zq^T?ZkUYifm'
            'Bj)vQ;uL5w6JX%BCDLdu4E>{lQ3%}J6=2dx@G_2JGa;#j4ZQ>&4zz`tKN1*d4)`VUN`=<q>HhvMSCrLW60gOqnO}nx0%RJv26W48'
            '*-+w2H;pYrdnj2kr^X>(%3Oj(1+*>&=E?)q4gM(Q3Yb@Y#p5BkRRRY;Lr#!GV}HU5XCO*k%ksGq3P3!oB=G=X7bHdrU_t2;B@E;<'
            '4TwI#`H1n42Y-6^`s@G}9~djpO1;b#fn_)mPm$te@fsgbYPs}2(9(Nv9GK#p`@zxi9}nKWIEzk?58fXB5FH-8coDrgd3kacy*Yjx'
            '9Uc4(y#h=PLjD{<iPwJyynOxk==d$7V=&bF9ns}b<4-9wc@x88(?g6h(=bE~_(6i01rweNl7)A74F;SdEdfC=OE1$|nxQO~z~0Fd'
            'Tz7!Y3W$jyLW2@5t`H&=R|1NgE%p!;vK*1rRu!3q_MxVxPGe@)C=^!6P6Tg;O0`6kdoI46ekGP^-qdD5x;_ZRFe>21GY}jPVOzdF'
            'L*X275c(FxeLzegs3U5k%s?%OK#niR@T8g+w<$3V3BnDtd@PJ(Ruw?BIQ`-E+q3BC-P?n+lh?1JACBL>Jvlu)IgDN&`~_C)5ZC4R'
            '`-6cd4<|>_n}f5n<F~KiODV^|H&(#zf^zt`cZcV{-0oi7eYrpS^6UG5hgFD9PG2MMu<iHOrL7GOya~Ra1mC~;7WPy1-7m=(;ZgYg'
            '+4t%tkyLv7L#W>|`1tKh^A?y>0c(kG;mt2&{Q7-(8h-!p!vSChfNb}-`wv<>1mOTSS_&HsS(e1T2f)Pz40to*9>0mRMyfRof8#L>'
            'fGi1?4?s7XoAD3}l>v1firw#EeC31*DJqO0(Shu_1JGKemmu9GJ17zEEYqrj!SCRfd36k%k;$l?CABw3;gA-N+CeU?n;D99q2>!D'
            'L;}_37Kfh!_{XRxfQEolwNesz55eG3Jy>g5pJaFvaE%gzPg?GUrbt)@l6aZs0JVh4Gf2~@p%?)lt81NN{97+$^yBf*(VyNS`FnHl'
            '_Tc5FtrmQrhQG`L&=~$%7uPcXXFQ+Fs*3o5^}ui%od4}VeyM&L?p%BkB1t4H=~1u()|5b-iJ-*^B8m7X=yJdUr}U?TI;S-Vq7?#W'
            '7i#QQSMdtq<>)PyCY%DOE3r1zZWtwkbV2P@4Qp?q0W=M1&w?5laRnlfudRV5c?q3q&5T!4CgYAy_CsB-7&}xutecVgzfm(A3D(S%'
            '23LvqF!1TO<MH^r_lQ!&OqWenTiqK<g9vhsh*8xE==x@7AY1{2067A2NH1Y)Xh_quQxN}*fq0-A5~xDJ)Sf(f0s`8>;o0#K{2YdE'
            '+nX?ETs5;Gi<h$`7R!lP4DXiX%d%)z!PD@5+G=qm?C{iXCW}c6V?gx|;D17p9%`}#wq!KfX<Lg`i_c5l)xi;%uBwgD<dtsIncrx}'
            'xF|50@Ut)kMnIJK>XP-p&q45A15DuV-q5Vdp&kOt2<;7`#UV5b(wnh(8>ba4;!n6%ym!=QRn6m4E#j<Ni$Gh7K*B-u0?0S>>&-k!'
            'EYec#GNcUEb-IF1LC6OJ4MNWqF-h2+bGE&qRVJ;`7>LLe8XSPlL*h$&1C=;#XV}5;=hyGf-u>YiL`x8BflwjZh;?!O+yfi*cUh*{'
            'GG4}6I#yeYaI^)ivC%TLm4TWWxR^<Af}t`|lA!t({HrVsviuRYh1u0}pc^tWtxo?4P?efU&$Ad9=3*UTaiWR=P{q!k`vVZJUAb7`'
            'a8#9a&fKq9^enigh-A`?*-jP{f}%Puv<{EG08bpD0Wl$~D&TU?R^wd|xN4wy@nv41bX{3;S2`=w3q+bqr6FjBGcdJaeL=ASoJD~i'
            '=RC5dLfbg853&vb3A~i>t0oAIku#(Ep)Ci*y}W8buq%MiRH#8466_AXQy5KIV-%n(>IUt0gcKlYCNdZ{^<ww;!!R7fSb`zV5<0)T'
            'cVFOe&hPFo!t?zLe$A~To(%mlG6xjLnh|yfhUQuCB>+wXWGcSi-|w3Xi)m>NlsHWy#rTE~HS=_PGay5NDh+Z+f_VL4YhQI+gAxkH'
            'psfT$(tL5+Ks?xrgbMMCbeY!fTeO+@FT-xr?qI3iNb1tthQZR*tD-)%A}!kWigK1FK-RW(hr1TYR3vFv5sZ|829?vk0qB~xi-!RD'
            '_2y;4ZWo{$Cz~1|>dQaBn~G--VM--SAWckvxVg=UX%Mn!2M33Sc+f^3iL-PmflBhWc7SPSA?w?sypGV43eflUj3&R%i`!g}3slqB'
            'pqU<$dU<wydKMl3<;{zOSEO$`hF_(F5SC6vKY!p;lT(tPtDQPVh9Pd`I<}`$vV`e@-U))sM?e+;5U>xKq+@YfWa9t+pa1>802VnO'
            'R>3L)wBxHfE>m%hXFX6r)=&jFg>5jMLo++C^dlq6j;Wca;^1(s&0iaK9n1%n6w}+KoTFU)k;k5{YP9_%FwnFbi~k~`3_?oLBmxD5'
            '0-k`gKns=l514&<lfoGAmDOx3(R_rdZkp8C8ppSwK7zjIR1zSvDvQ5EchK5(nrCRLBGAH^1ujdPrR66L{zW<~W8h`*bEl`v9sF=i'
            '!!)CB(Fqvqe>Hj5T*FMEUR)6bozeU#0j2n3(v;1z;nn<vL6YeXgs6DEfadd9%;IYi%$hPz;_DP-hf7(4m<0uZ*GN&vSAcs-T83uK'
            'cE`hTPe2vQVTrC`jWPgg*5n}CAa?#pZ%&SOpcfe8yF*ZE1RinC;wpm@p!6nj&Ikj7Sdz&dD3%nLWh2({N04lE5s8+iBC`W%ZB|6a'
            '%UCoS!i#!`3B{LL42UE?!V5I9!4}#(dUd)7i~a$`jiPMw#sD@PixXh5CA2JbO&qpGxFp3x-D_B108uVG`$ID~1(IvKBLbz7faU?}'
            'J`5z(u(?djBxUFT8OC$#ycfkSjW(^9bfAnd62ThrcaO=0nwVWIkEA+V0Z|gRRJ=eC<6r@g#4Lq%hwAf24HW<cnrHJIzr-JLE&5>K'
            '(?{s!5Ar&b`BJU{F|f@mIuTY)#h@9dglnLqF4K=#GOa=`1d4O~cZHQ>KD;9N6ebl}i7-NmFwiISEb95@1XM7qI!g0ZQ(N^Bod=98'
            '`$m=s^$HK#LG!!aofl}2cg@gjQ6~)nxmc;}S7^(xnv9hTY;!DLk#25uc=sF|WZ9U`ofV3ZDhf-EWncy?M}6mJ4F}-`WXW=3Px<UO'
            'oFkAtnkAI5Q+CSu$0rjKP<uY7N%z}TTt#ffts-rThf<_CIK|kmqb2!nGkA&*_)Z6I?#KIw6)b|qdAs!s6r^3w+O6Pphov84*5_VR'
            '7j3b?S!PdKFF^_~6S9#3rM&?ZWGHq}(peEbLm-`H<k}6jh8DUy34(-sIUlM{lHtXqrC4l+&<3gO*TKj(b|JnHi=hbacx<Fq>`?zZ'
            ';eFWds|`bU1PZNbu*G$E6@9e5AXqav)SHbp(!?w7zV6iN#c-?*l2I1uCfX&at?&V4X-lnm`UUUo@)@;vx=HhX%s$W$|I!wqHPkJQ'
            'wqSx!=`XsY)TX;c7eQ-ej{%s(^bX?tko*j0GR6f#JCZkVL+cYLECF!^>GglIJc2A^9_K}#;!&n9!dUTH4ph=Avgpc(D3*C4N=`iM'
            '58`O|lx$3Ae_VTvW|dVChT6^pJ@DMNw(ix3zj0YtXdw%R|7lG7+sGeqJ%<HvCV`ty06c#EmOEp9J@-~Jh{X=cmpj5=NDx?d)MD<0'
            'qAmu!`Pxeg(pgMiYv?EFO91LT%{g8M@l73Mu`!EA`5aGUl{_lmYl{6It(U|uNq%@w8{j>;wUHCn?bVu1{89}J4V+}0{;(qV67&#L'
            '%!W*q*b170(!7yuujG?|J!W1&$GREM)VUx^8@E>liQj-8M&w{4`P43zUh;v*=p2DWOB<#J9f>OzrKN*D!9sf$bp(Vy`fBP_&5SL)'
            'rSjk~UtIr>uF#=m(YCnZ@$s~G%Y_{_y8;OKxINYohrNMAMj^kR4%1|$yhBIIVs`|nm1fll`5pNNpj9qSrrgt5evtE~mVu>#pvZT&'
            '>E|C2?8qOe!-ob|$@VyyKUz}17#g*QlpC$rh=A3<YpWBo;-@L?j5LOHjYd~-XHe_EmG>Kj3hddj;-Z5H%VR4xb<!I&fL?UuD*Cj2'
            'CT)%O*NzJsqaM%qF9z-YtY`<N&dNr1O0=XJZ0UU1l4U%(NEKozsHJyJ!(1%vRPPmX&<Vyj>zSr~Y;t=jCHfR6Z&X9xH`R;Nzx{;>'
            'S^2|vQ54$f47x)cNZ!D9R0HNad+&5xaEVB(GJSIEZ58XfXtYiC>99iYOjM&&AG57CbK7bn5b)<f$uB;g>DI7D@|+q#&IT>m-xpnO'
            '9Mfk!{_%KnWw@jWG?qs^pNi=*tF!tYaMdH3H0bXQ$Wx*Bk*+4}9uNVY>WN~`K6Z27Ft@-=Lvip&5Pw{$4q5AM0sMKfkmQiZBhyYx'
            'ISL8LQI)G5HO2HOWY-Ke0zqE;5*BkQ%`DLcG+tHo5g`cem+P@O&{6<9yXW+cr4%Je)WwPo1jM>4KDU}lat)~g^;flb1kErQDR80l'
            'pv)bXg&XGlVnqK}fVJvC5q(B9I-s$^!HrMXVjWKZFxq~C^2*a0$1C)L3$#q_me6v}z&kVPp^7lQHc<MhyFst0SVVC~4${$P65Kn8'
            'c2DCtNrJv%dAbqx&^KAD@p#Mp8FZhM)7QJ`8E1X|KtQP259v2oWUy(e{kV+}bzr@@mDaHUZcr;4WDRpznW751W9DOGwS#xwoPGWB'
            '=~t&9+SBo}a>E-_Lz?;!c}o|(fw?w1)UeqHb%Z9T(b91=Pz>c|Lv9IeF93J^oiilth~AO@g`P5-d=Zs~=C)=OHMct|wz(e^@NoN6'
            'G4?z`BCSXOx-_C3gAgWJu!0Q$?*PwX()(P9TpZh_*p=KZro0~81=VY>s9J+|QT5s@s^&ujeIkDU&DZ<;;tMsTC*rGb=uvn5M0|sf'
            'svWD%Zt~I(+{gi=;kHrDIG=)2`Qm4!-UidYor<lVswpd6xy17(Jo!QQS@O^|F&adQmFq0B3XCE0-GV`TCI4ToyICJ#?WN1UzkB2e'
            'EI_>~aLH@*2G#}5DN|BN^0!s;$_hixP(v-z!FUG{pPJl(#fdMiB5O8@nLU(LwjOs6xG=>c7M6YZsn^%7^a!Ya2HJOo-g4ys0>G||'
            'Wjd!2ELn}j+1ZQT_%<#<p#-X2HmiCU<FIzCcp-mt8xPs~#32sEpx}Y1D9R5V3wKQz6$XW|lOGS8bbS~r`7m-cRKlNjQDRl9Z9Kmw'
            '>Ufpbdr0B-O1VIf@KE{HRA^1WkPY}<Hn8qVk?$a{-mQ=`aU1~0YvmfLF+Sk-3eW985GVIVY7+ag;yH^;mj%Ac<d_v8J`+h~CUFBJ'
            'R4eqoLlnhc0k@uOn5$W$42AKGupA7PwVD_YnPSy&UhGo<D}-L~(20(x^A>l)e>lhv<tmnpQi@@)@*M9CvVGPV1zA+uAu{wVDaa8p'
            'CRMK+&k?Cm6k#|~6b0yLZT+mu3B`pux;uU!o6kTZ@q1uigc}Num)GzwQc{3VHANX^1bZ8GwBni`TQ#*JbvFb=z@wx65ucez{h$`n'
            'Jga4h)XdTRvVZT)5aZp{^fZbplOB)K_yHoJ#aY=&|C@N1r~hbV4%ozDlpOhhva#`r4RotcgCQ_DfD(o7I<3vi^{Or|%XoDKE1jg5'
            'c%)_|eRQ#&*TRVpVSASjlC3cC*lEq60qYe{f4^^3HqtvVS{8+L7C5kvXZyRLHn@#C%~9M3>l?roVig;lComTb|0$X(Ta)IHVmNT&'
            'pdn4$X$kQZBzpKa$M9FCr5r7(On5sLS4wQesg6b_&0_4dhA~3KDedg`YM=mf+E*~PMfw2=`Zeme$O@^ct7aMJyC^23aMT)H4x_G~'
            'D`#5mnoO~=&8naSZ45b53vb7w7Hq0RH^;CdUq1ULSPbtp4&49pVSoS5;KBWH42qub^>!sca6^uL3h4oPjFRg*`sS9y^S>SJ{x#nH'
            'xWD^9qVetp`cy?j7st*g-|Tme{aX8UPq>4N?sP}+lLv7>MsGcbBB1%;5xWH_PMvY&-oz7y9h%{UO;Ah(ulOonQ*aH+d&rn|<Q3Az'
            'fP%SA%UVdm|1Q%TnTI5yP|PV)HdFLDBhD>)ww8`$Ft$aehP#toC}Rxg;ME6d14wDLBLeHgsVxHJ(^Ibjl*$0KtIbSKTWo{Z{_t2A'
            'J?Kl=u$kkK%aU)60yeD^)22T9(N;;Z;21<k#NMFo+iB?Lxu%pC;)|(x+GpkSV9;^nWV1YwxBxpuVmm}U01SrhJJmRh%xuy1gp)bD'
            '?z$eJ!J?%Y6AnZ%?jjwlJVq-5=v!=W4RkZ0lIp^)M*6V3rwfY?W9Wg(uZJC~We%RwD)}-cov`TeR<uT%34?4J=%DOzQ=^P;Arp={'
            '@KdP^K?s<K3dMN8?WAZM|IXQsopo{NXqjivG&fDvZGgCPtFEeJ7mi}PFDy-|9!pcOBJp7#lApL|97McxkZ2+{FTmL1Ur3&`Faatb'
            '-`m}?Tt#Xmv>kD?M5HZ!pzSi|-{fX6hQfCR8Dm#z5`_1T`PObS-6HgO`^b$(J1c3HE3;2Vg5X+V!hyJ&r|DE3W0C)1J^j-5Y~@3z'
            'Yy3Av&Zq>?BIf|wH?L1m{z6pw&(KnFTYU!Qh8+5g@c(&K7B}A4XfE_znGj#4K7gh{5mako@oSYeN>~wy`!*ScqW>RdS?C~SmG2@i'
            'NS`I&t6NyYWt_;6f}QbWvz(z+hI+y==#DQB&W=y1rCD6DV3ic}1|!*#S%PAvBAGD_NL&*qrHl`?li~t<QHsQKYOBwwa;MS`lq;NN'
            '%3RrtYz8f>Dd&JYxd}0%ZfXYKc(d-%RTVJ125i7<5XXt3UZd(_1E=r%GzNn3gOyD0Y>AAppFY~sLq+Vk&UMiXO;zYM9i7;f-w$Cc'
            'KkeyJjr7yH)zj<iTYuX!0E4X&aR`Pxc8I-q1i@w*I5WP!5KkOHQcu?-@ieqxwxvVcXI(^Om@|D@zXqNhWp&W2-GZeT**?;?GiQ)@'
            '9&JNsEj(fmI$UWuYpi$g*})GlPhK6JynXIvBZ${!D$rdu-laIDDkWdziu`ba!gd=)3AmMc0NPI}!9Yfp(Q7nmswQi+|MK1?XddvD'
            'Jeb*mxu@lS(WlvOF&z0&BKStLb)*N4oJy~eTEJEncGX52LU|*wjILyHqFu?N8&}6H3{`{zGSsEM+6dWhL1Bl=1VY0$)h*sUR7odB'
            'q0p8n!!Q=8y){3=_HmSQ8Kd{|Wuzwo?JnaFUEJFrZSN1{H+M!`mQQc@2i8y*aVp<`4yQ5_jz-MCcA3g24)ZGU*r)s)rz}b=Bl1nZ'
            'Cp;K6mWju<x5<s&_@WKU=}0>=M~he)&K?NlHqyciOm+t9-v(Pxn)4GmQjzVF^ht_a)Y#DK7o_?iCfQ*(E-$GZ9pYG9NhcuDZ6-t`'
            '8acV3S#qRUQ{_rAYJ@6v7<naf$~jAZvF^A%GL#F*6qM!^GDAr|lmLWtGs>(|=coLm#kvf196NzIA5t}Yl_r|OxWj67%N-8FZMm=^'
            '8%hh<`aGIVO{PH0Q>fK9V5Z+KPqf-JQQdy~bd+9xhRyqq%n`dFpV7M6RE3{ozrylHJ#IFg^Y%yVWqZ=RY4>^-167&<y>VY}-Nw%9'
            'yR>KEStitotqWkKWfwc>(WW8UiEv>3<Z)hZ9cQF8ncChQ2B4#%2iEn-^&O)e8@>Kx{k~%xj=Z$L*Lizj7w28?s0-tJh3dXR6oVi-'
            'HCy#-@Wes0M<5Dk#zNEiMDWgh6yiMC=9VNM#!-oZY9{NrFp-CgI`O`goN`1pVcx<sn|Ao9DwBC&nhM3Y?#vvE2-Ne8FoqD31K^zl'
            'y!&W%hvo0RNQDPCL(hij$!FzJzpb0XZVa)irxqNYZQ}K$_exJtckkvgm6NFhf)N7oEiu7r`fPth@H@lar>lDOkhtb;I^XBdEE2Gh'
            '8l7X<!Lhq-sNf0OA5qsLzsfY<yk|pil?~4L<#Xusjh@7?$6jo}>rH0{+RnA-Dz7JTKCyVF_i{wX1+beuSKd}Q5mOCQ#`QJ~&ekib'
            'qL^*EiLjprahEw7;WG3(R3ihp5MWt89tk&nA$sX-h>!!N>qepj0`6IiU~`Yv)DqC?>c$L*WpT?#KkQswi7Sd@6qpIJ+EW=ID-qa}'
            '<na^oxKzi#BA(H?1|Dg!!wS_8$)tt}SlRtud-E((nI&W!#00qd#8wMi${BXEgDm={+_6n%yDzF!SYfMIyw5$h)qd0Mpy`%$vtlBt'
            'QHda^ivFf>*YmWPZgs+fFP@m?`1qT2pc7|+@EE$mhsv2>r$ctVi=AR_646#+_mf-2OElW5W4;~KUL7yoz^kFYoFu*FduO3Oo^>YU'
            'B$AvsJKJ*_)|moBx1y8!bVEffVJf02S#e6WW6+C;_5!B>M6Jh}jF(DvLkFtR=05h4l=68e&f>LGvQ9jF9QBmMM6DAK6<wUc)U9ol'
            ';dHJ*wB6-#a=W${M}2B7Xea$^2ifTR3;mh)SYL^QCP@oh%o|iYo9x8jkE#H)J)L4U)J6{d5$e3Nt$NNEZy)$+X(x(msExoMPx+f$'
            '?V;SYlgInaxf6HO-A+2}L&w)x;guOhlrZ)CHG*rOAh=o3<kE!f9VvD6C=rz0+YRUYqh$9Em*J_&pxB|kKQQM}EtVKJ{%l{I904a='
            '0Xr(l`7h3_LX0_+$RBXN7MBeGJD<yN#OVTcbQY}&>2@jy3>h=f=L_w9#;x0Nx<a=>O2Cu>rwowt>>(A=d{t*|)oxk#486U8?m_Z{'
            '$9l{zid(%Z1T`z4<M;?R5ycC#<9U851CSPdk$eOIb7YvAa+$S~l+(<_ZSo7Kc+o4c52NAu??svi6~ZwIs8$&MEAqOVX4Vtbd-1JT'
            'OrK3@9326joA$L>9jIlOxG!d_QTpw0q~<#dA1`k|N_cL0&%<(S*Y%B(Wml#QeG|^-Is5!ku5XJn+2vzDn=vi0H&U=onPgi6BL2EE'
            'iFz@CGi<=m63rV~XxWU>l$|=De;vx!rwW_eiQQlreLzP<?LZG~klY%vVdQM*7jCl05~I+DLT`||40NS%T&=PcnhU+aPyCO!6&4!L'
            '0lteOO9D?xqIOXd$C6|n;YbW0f9yOo7E!+~7ZRCMB!%%oP<Uf2YjujHT1~DZpioXIv-Y2&<<@EaU7v5I+HsAIW*kpDqYjhJzBUL='
            '6*Q?G1BQP^PqQ!f_b2=N!_b2nw>GXnrum}i`cQivs<Ou2fa**=;@qBssuEYaRM0K{>lMprndW%DL(|pxOH9_mV1Dj`yK6R6NSzh8'
            'BM?C^kpVyls`ZfU4f;X1M#a9JVx4YtPsjVF%y(j+QvvUf`4l`t?Bv!iTnszsF2pXjiGD%>)mM*LQ{_>%wN;i<KCH6lIcl{f>R6k%'
            '7I&;r&=PqDdaF8yI?D9#JUg1vqV~R^;ek&ml<m&;+?9yb{4YuRD-ziw8R;@A)wvO?Hx?z8gJHsDFi#7#&DRYK9ySGz27MF|eIL<F'
            'oi5j*gy+&bttJDWo0x~AEI<xZ9_Vjk%B-vc8d74Vn0YcYmW2uBfyIkl>N4gAO-BAWC+ot%p3&186KC94vHEk)*2E`7bbyOV*oy`*'
            '2v@{C(6pZ83YTedHQn5lhW>$!ql@7Y=Kry&mULK&8mvN|JNm1gr|9^Xf`&c>Pe-)h(3Xd~qJu)AB3df7vKRl4vI?dnGNu#z=^vDY'
            's9d@l9}pQVO-tjJrBT5b`rD`cA0UQ9ffmj?MXfz@IN^yo9}yo<2@t3_S>$Zx9&&eqOY&jbXsgGMhwx3Mlrd#(_09(AX+@PWN+mbe'
            't~qE+w8l@Z;v8aCzmFQqyI7-gVgVj^QsbW(y1;XNPP)=|`_<!*Z@6JC_C9&S^LqGz#!y&a(f9;0mi><xsVqH<8{IGBhL!7{u#~Cd'
            '1cRf=p_OmamK%}rG-R&Rc5~G&`<&idzm;U9UxdBpWzBf-9ugx`>Wfem|Ix=cPW-kFN|fX-K}FaUe9j)DwisxCleXmKlX(hBi})TW'
            '3aR6$S(k$Qy7XHRu_+Nc8yQrdWgxeC#UU*tbyrB0UgnthF`#^4xp?v7C1yuTK}KbAeu@(pw-Wd>aye9Zg}%i`JbU&#I>RR&`^){`'
            ';a<_#*@&n7kwc#pFM@{>mr%s>vZyfjfpm#ICd=sOf}u!H_kTAQFO_{+-J@akGj0k7<%B3Txv>u^A3(8)XC(z5Cq5GZXroUv{sDyE'
            'tb&IR3v?@(8QrJn?GJWC#2u2@z<P}<{3rBMp*Ty+0anz2+A0s&ltS%=fOs%@IDERK8MBuTHSZkMXgZIfvprHio+bDuV6;X#(zWX+'
            'PV8+AjE#W1Ya8F>8BY7!4uLlDQ^#GLE(}7ltw-UQg6$CKT*EEp*nMp&bME03^i8;--><o-NX=}2q{c(Rv@ioRb*sE_o-VrursU12'
            'a^kyEZ9sz_nNfxIE)B=K#PtmKu9AX=JALh`;Su4(ABax;%azM*?1wV{w22gf#(}sxln8yQ%{uQ?D7%jehbGgqx=#yj%XCKoMp|oK'
            ')^X{k`?PvnFjsADX#xk?tjLJk^2hYArDFN0?XK5%-Pc6PWM(M-U10m+21h|PVkKC$dum@8Hi3~hz<A7?k@za?)bZ}h(pCJ|J@wjm'
            'j_Hc6_apI*>!Ifm1Vz?N-ULv{m=TzBayD>I^;sENLtwuHR_n@6&OMt&#NBuH+{n|@9S561uPY68eQt|ULO$;!XV!CkUR+;fbNd=v'
            'IFRBg)%qOoEgWaX<<k{T_`D7Mi?DB&%1On6Y`a-%tGyx@5445afHxmSds=g3ofrc?)bYvmsjcsztg6j(>cjWmv2>dy$uCdjhmkHT'
            'bI{gsI)SM*>E#Iuoc;#z@+gt)Fcn{lCr?DMYwPZb;LESY6Y<|2ZsXs9*T49^ST<sk-GQ$ojMLR}y4?pLw%3NTGZ|GoXpQp=uX!BE'
            'm(~%T<SV)s4JPYb#`nk1p215GZO>^TF1&JIK*=w$x&mkC<5O&=g{YevdddyqNPBGAn`v(AYC}V@dIk?2z;qv`UVbb6W8c`0e|e&A'
            'L?hd#TD><c+RC()o9eo^73wPV<d?ly#MF>k<bHu`{hg{aj1rXs-nnku7bDS2p+=&IWuvGvbdM=Zl#DCM;p~pK#}@7U`hoq8Ar})D'
            '4bIzFGLnJ~KU_6R*VgH@;Ipel8OOp0)u_5yMLDr%+9xBO9wqWFrL0Ve4Qdf3YiThqHeRK<X#1&ojLXVJ8(9~MY#Z5ETOg;A;GHHk'
            'B6Q4E1lQwScm3E&hx1Ok{!-e-qG#wGcM&V3plot}(MD}0M9~3Auh2mAIM+>Fc#3tW6gPLFsfaT`Uup~aL`7{FJj?P`Ee0Yior0of'
            'iW6VLMJ!Xi5Klu}L-p3sFaNDJ*O5rr!L~5mvdFYwY0R9I$C;?Rp+lulc-Yt)<;9@yT<{_MPz|`bv9=&etA<K!xIt$2vw(VV8$B$E'
            'D{0@yg&rZljf3qgWhbOrk@ud3tE(P90<2+Yterb*H7Q3_=qzKa?BlxZvX0IrxQ2gVBphK8iVTRo>Zz}h){VLDz<X=mTfc&BOsIV_'
            'ectI!bEVEyS_tHcRbCcI)j&VOK7xZO`hb7PAv93-b>baic(IMmVZ~p^;?yZS-OZ^=GYBw7OSuyKX2(=r&z!6%u4hmx!jxWy1%AEZ'
            'xM?be?@osgpEFHS4jVdWVbCn+rzFGWcs#z?a8Ga8IToYcLBg*d+LzC4A@`)F&EIk1e7^q7`Jx4fKB++C&_?OtVUz0SBRnYb&2Mt<'
            'w$<p{bqhXHs8z9yUu(<hu9~G1U{s1KWhJy}wV8?1eZ_Q<uyw{LD76)aNUj+uh+iXheK1*qIf;)x=emm7-j&5OnPuvZO|{x+iM`B<'
            '8J+d%Junt0It`}^xj;I(Z1BFo*7+QH8+Ed5Y<G^*#;9%1AK}2WJP}d{be9Nv7vO15pgzI~HG=D7IeRISs(#K!Ya}ZC3op#!kB>cQ'
            'yfBw(d5!-D=Z}7a`XI8s>8xoJ&m^};r<x;6w-F7X-^#G=A3f?OcPV*5BC|RUkkm~b>CB^nFdPp<tb>U14FQt%R|OBeu>n_-&|}2F'
            'V+QW~_Vldr1;-t@fVk*A^sx$ux);$%u_}l!F>y4y<B{zQM>yG>_l6JX`n@9D|9YK4jnAFa=V<fMuN$z`A@8Hksiez%Mm_0T+RSI<'
            '9T<20>3lnc2ft227$btA{svI&WT$=50Pg5xgUybwq@uNyBQZ3VVEw{3FYn)XS`7OKW)&rL(dv3{IR#apT#|8kQ&HcP<-u)hqHT-9'
            '{ThhAIf>lBTNxB+GTKpLqaQ-&B7v>9nD}+Ib;UirrZeyM606V+<y<`NbDthOZtIw?dNzO>BulC8fpo>EF^UW*&OP0D#;k@k>R>9e'
            '>D!Sj%Y=T5{G%ffYYld*RLliGQ7+a(hkDm*@6iprvfSta>CO0}xnho%0SBGSVuzUc4|G8cmrhIfDo!pEnJZuY9ljXe8JFoQycaPr'
            '6s61ENI~CpullQVV?ok9Z{^frO<z>kMcDvF_;?B2JWJ9NmjZAb?=Ay6xRe#<?kIt)r^A-bm2Rf6c!TpM=f0M?%FY}bHhFL(N(Z%Z'
            '?n6pC0xCIOKBuN27V$O6UyC%Wkw%vo2ZQ%ZC==Np&-Xgc@Op-!9iw7y-T?iq;C77aQnk$ktDB_}Z1f<@%O=C?C}%MyNX6SOiY#gL'
            '2;6HjekCyYzDp3SGY~ZBPMEni3_3Sb3^zn?wc<ra?Jue)Q$RhtPFU#*FBFwgS&&5r)3`HZuV4)GxO6?a7E&fvgsDIxU9absI4e+d'
            'zv{t@GQm8!^(Q78^BmLo=69U>UBBUwPhZ8*6VudLt1r5B%=kuvK!b)OPl#bxfS@8Bslsp3uxDZm{DfS(SEez?!9Cyl>@Th?$Vpke'
            'SI7ko@l|a*)VO2uvXs#&Y{*Z$|Ba+m&tfxeZJ%jdb~0q$IB+@4h~50u)cXAVh!uAn+f@D<6lilwrGwNIpTbQ=wR{I<^-rfLI@7D-'
            'ygHs`aIjVQLAJ*tAnfvv45+QieKx2e=U8o7L0hplgRW&=F}+SaWMvKdRr@w2{Kh~0fG-vKVU|>>*8fUc3(g;+!j&J+ZDAv>IV}^n'
            '@TmL+TU!2?g37wLsMvBBPJ}gqj`HHv#fG`Kks!<h$ihg9pz<<&dwlx(#ZSjatxM`KwY=gFP}g3mqFZiVWu`2Tw8kOkRHk1#NCA`B'
            'R1ICN{9MH5va>4?7EIcCHWDbqRG~7~vZFMJ7!TcCW@6L)*bAF}z&T-eE}!hZwx}D)?_|O7LoK}aH|1yfL>1N_X3LPgF~`F;Epyvc'
            'Oiabd++9UNgEn;OivEpOJs{n&<J}3Jd>5`Jvmrsr;PF6eCzj~J`%i(D=M}I)je4~9)F0iq`MOHlHOAxj5w}E{mD+warn2UDJviG@'
            'n48>h+qWuNi$VJ;CAvn*W=m*?nqf)>@6=PNXfzNPOMEX$?Euj7hVG9Q>%DG=Yza51>#*0<o>Q0uRrvH7;tXHYJ<TBh*D2=1;+qA^'
            '>T^U5Qq$IYtR<spUfocJT)tIc)Z?Mh%A1LIOJuirxeDKP0W=#>dWLsf0N#Thb9nlb2(Dy|<~>|Xl|($mkeJ-2+<4DD(Os5!JJlK^'
            '4uIm{<`sH9FDdyWTAJCgewEfLbR<vWYOJg@$5;_>DIwFby;EdKmsxOG+$bw`d0w)>__t(?RsM)#y;0La+Gv(}HQkhRBy?1BGHrOR'
            'Y^^=exsu21oc`F$7m~%eF3civ;@}S)p`kZ%#7oPI&{@y+el|00hlx7DjE{#N(<nWA^_M_J^l~$xbkWZU7Y0km*7U0|{?Cl+HWIiU'
            'cc3?c;7u@6cebN<KOr~Wi(>Gzf64SvM&>Z3i$)u!rp2Ak<m@TpBc_M8EPF5BwmEK{jkS6?^U)Uz_s$3-lf2>gBpv@1$kIuj|Co&4'
            'Hk&tJAFwr(H&0~Nq#62&=!AcuiRS4->NHtw8BwV+Ju``Yr%V_+pQdBJZ~`+`nnb(ytfeUh#axZp_fg%D*z?ITExQ#E2O`S2U@|?C'
            '<YWU@Yz8R2;D_c-chtL<qJo$mWi~hE&4%I^TMmiis*-ru2phkE)1RIloV`1ZemZ#Z?)Vf1)IX$Is!k0xr0@90{r}%qE%(v2AoODi'
            '*^zp`cy;@5h^qE|aOc>AU{J5`b=!zw3Oi43t{a1i%96(qUNRjYGZ8keR~{7m@RjrEY(!+0pju%rWd~Nj-8fVS+tF@#O?x-EQAxdl'
            '*3&_&Q)--YQoZ}O8Dt4XIB1Dy8*JcMcB8m4Pv<v%;Z1ygXx&0dy6G!;;Rf8BPaUn}j#1X$q76c{dbl6-mc!$C8$M-f;5m+c{W%gV'
            'BWH21a}795P~D^;_QYY4#j{;x8oNg&X5M#S!HV(Q&yw3i8&Xzf@prU+5n=>*nK%xBO_g-%TErQMqXdVd-|JdsQPhfqK4glbx{+T='
            'dHU=rudY2Rv-fgMQ{FrH>Fv8W2jb_09}iAX_I_r?_jO$?DR%E>f7tcv(YFloefF5zh~Qt$u+wUy{c0jQGSaFva$I`jxGGTr4(>A='
            'IKJXpERj$Ez6S`2a4cSO2#-qkPNS!`=n^Wr$w&j0&U4xjrQl1tY6GQ+vAS~(_zy?$Q&SpY=mxrVg<->V1qlft#GVLi_%_AdYe0in'
            'DfU#JurOO2smh<%lv%{Uxn6$b;2&_Npi6<d?3f!+!oHeyCFm`;Gid41=EImq6|}H~uX~5pDJ6Z^(s&o;?4-PlaUXR&zbetxzZkyz'
            '`QYuz#9LrN?FyNsc;J|%BDjO<_u+G~1|d1yyOi^Eo*Li-wxPvJT_UhX^-&$J<CT=|K;48=6#GsW>~*(~bDn9oY;RQqIei1Tcev-4'
            '^qV&Va4UBL*1&^|*b^e0t))H!CDU(RnWVk#dP?JOx7e`5a{YSr?!j7}t?bnHzInOs++{(|>pr_F7>LQrH3g!2fEe{mcjV4t)Hk%y'
            'zwoF!hqf;stA3aqI#5wD^##z54aOLwbz#Sf$(#ER+ZWApCJO($bmJPSub{+u^FVub5$A6uf>Pa&p?fdy3EiVe);u~qLjNhgxJ}b2'
            'gmS_;_Ps&3t;^e@MeLaM-a>cqVebfsUO1($xak5mlZsxI8?-YJ{XZ!unkx'
        ),
    },
    'viral_prediction': {
        'sha256': '18870ba7aca2895774557ef7e2c3542fb89fd10476f857a2b858de31367800cd',
        'encoding': 'zlib+b85',
        'data': (
            'c-rM!+in|25`E`a^v;Wh2}#jj_+jal0VS3b0kRZG@*0L944RxK+2P!FdPXv><-hN#zRX>=vx{IKmWU!vPjyvw-A+~0p66YQQt@RZ'
            'm@0T7RUubAVh=Ls(KO3NC|89{6Sm|^L^2U9N;h)F)RyBNTkc2mSAixDYT}v9_Q{g*BxLh>uusdPT#A5~MH+K@AH-Z0A~$5S(daTQ'
            '#4=5HZ2pSzFqA~YqiE04^;+Z#j98kPMRsxVe)^!Ml{%|5&Uh{rRP=WuD@KWBHjPp`h?2=lFj+9E`H0V%$Ym^&0`DMenkEy5Ev(oI'
            'mgQ+!n&W1?*p6S1XqB&u62z6nSQOhdRQh<^GzxJ%CLVZH>a&3nFIXs6G8CFAB&K;G!coskzmw9Dd=P9aa+p75%e2@sXiaJtLK!Qw'
            '6hdX3MMA5sx58%PLFBB=qLf3IG-p}4jHKEMQfyyrWwOD+G*QfVw2G(VG0k^cz-;7s-e{etF$;n<9F&V7ASuul3}6v0NR13l$O}=x'
            'Y>vV`PZ<6G3p5;!?5m`VvpwUAC7D~wU@9!&zbqV$Zf;L+KimZWIKB9AcEe8C(}>~YH9g}^m={9#D9vag$RfTHWNVSrt6V%t@fhUd'
            'pQVI3y@}z#GdS>po|z{TRj$`^C22df<+QlwGBU4gPlF2&_h4}tjozNUJN<BR8@xZi41PVoxxKpn6kK0@Bqd+YtGB;iUA(>ea2ve4'
            'KK=Rj{OS^K=d+jX&6dP&epS7^zPh+Lzx*Zmcz*fz>LXsgn2$!IP^_!)3RYW~UqnHSIN^z3<Xk2}xL&|^v$r_NyPT8V9hnTrZ2E?+'
            '5eEhP7rTTj7TO$+AWt6^1UWu;)gNBX!n<cDCk>YOnuA=on9M|+6??;+KDj{DBu27dbJea{rDalh_cMEl@tBzDG_!Pu855BbTIW%Z'
            'V^j2VT)Vc*(+Iye@aYPE{|?(t*rQCs^l`ytQou^Xe)=;kJX6SwfXaqYzUPuR40z#<|5w&0qp_2=*aA7@8zYVTPBMFXVh5cgk`3SJ'
            '8W!;I_Yw#Y{$EA6p$>=xsr(0`oOVZ&ig?>OcOI=)8`d(467J8<1}En$5^|vnt3L-)WV3`PW@p1uIqH0+t8a8THmt|a&vbabGJ^>q'
            'e>?*K`D1hX`D|{Z21qUd#f>n~g>(K_{eBIU1&E;_&gSe#K-OnJmWl6ob(s!w8nYkRs|mQh8c*1iel<h2W%wVgC`9K^r#ENF^0VOm'
            '+3l}aZyQ*&6=kkFp{cdr)bQ{Y&(-NkccU#uQ<EXhjqM`U+gP7!>e8gSdmxK6cIjOEBJwm)d@TYM^Gx`rbx{*WU59!ml`IiXk09WR'
            '=Q-ckJHuk1i4(oc7*(@s6YHe`(oiJm9SrGGIi6$q*RB|~Om=X!1F6~ahP@EezqvG}%%ayOu6ijt3?Go7<MSmv-jsGnobC}vRvYs?'
            'MpSFj?(qj8^W2p%+ue!4c|iqe9Bl9BZ{fEz-w#wVOA)Mdu6vDzwSJ8=-Ckl+9qub1B!&~Nv$_jj-3~<iotjQ#Iy8PaStJX^YR#F?'
            'Bi#rwrJGeW_N<9XV-k^?`gUw(o8Unc?98JWwVQj^%PRycmLDW%?@!N;O~Op^mwJEt+u2O>xz@f@?CHqW(4z@Ea;G#;XuNZ|Qb!B+'
            'WP0!*ftm3%o6Qb~gJER)V(?JVX9_plejPxM-a1Jcl$9Z?+i*pbfujBz^F}NSIz9Egnp-7vM(r}KC^8*D<TDIT6$*CoBat+sfP%+='
            'fe7ome~jYPC7rStCjQ;A!M1)slNe&|7L`ncT(wmOh%4H~{m8<OJ^>ZCgDJJBvkm)A+`~G7lY7paVGxN`Gwp1)*Jb1GeuAIsJ@F4F'
            '43SCvYGsL2PVw4g<`|MSof;L~Y9?(YAGda*IOxsrU|r9epH-R^GATt*o|<7=5J{+uLYt{Qm?O|CG-0!*wzWy~Ys>D*jd_iut~A-Q'
            'M8E1XN+D6LxQaO**Fr6tb@}OD;XmW3u>#z_fgTMFbVS`Xg6O`&PWSO}V&GN|n}t69o*>XMcvafh@7#)}X!Fo|1lg*nsP)Q(z0_TY'
            'Q5?=EFHiu~9aonTr}^1>aG|r%9IB=}G%@7Kb1?|&Ivl+^nz3F#=!0paVA=lVFnL+ZDC}L41UxS!bs}Gb5Br8j{D3ZJN|bt4<efLD'
            '+lsJ--h2;yP;^F-FlrdK6N4$1>$8~B7>U1A(w~HYe;fn9lWi_k8a)8=T+PuDbD|@vX*Rm`fa&5)zv=_*Pvmr^?|+7`eW*Z?P4qSZ'
            'WH>ESw0_}&Z8Vami3!K_ajI27&s2+N&TDy@7$N<8sELN=Os;mCLUZd}VHrrI?NR!O;}AN}9DrIpK5~~TM+`l)hQ#jO$&rRxv}h?<'
            '9ChwyYT#@{m$53m)j`A2U~mR0HHN)qo%8@0zj3Y8zxA<=DE<uUS77J&)p>K$%Yv~s;*dro#~1xPPc~FOpFF(Hsl1qQ@!(GTaB!_`'
            'Un&V`kN+AqYgC8yGEVGuW3_zO-gDocy=pM3HN3Ox-M5;C9v|vy#xto>z<3OB84t3eqkW4JMz}U3JIA8OGIxu*b?92AQS8`XPMDYO'
            'y!tea%?_T|-bSoqU;v_4vVFNjObNo2<Gnz?q2w1dhy(TVP+zl9Aax?-n<Dd-*s+#mD~<V2JtrU7-_f#mV(Y*Z*ydx;8m5}?EPl`>'
            'NUB5<m}H%}mKFBZPM~^}tYxT2;fO9vb&EH#(Pv{Y!7s1hH8z39dB$2|DB9c2pKPV!$xzzWy@g?6Y)&1E*We%XM6OYIOi47PCr3{`'
            '4Gy}*6loUmn3JLC!e4%=JRC|R5cd0qH~(Ap{T7T22EucWW<tr0Opx3>LT?!I;Ycw-ftJndh#CrFy7xcQR*0mLu8o9_jEC~*P_wWX'
            'JLryTTQt72IQi}ow?EWQdRSbSj8@Fm^6r{_R6RI!GoZ7_by)Y!{#zqR#cYg3>vaY#I}WC^Lr*0N$#&(c+`%^;&dxKhO@D_nle0~k'
            'mr+qhB}*f|l#$H6)^N~0P-jMHrnj9Y`$b?U_9qDYCE~?QyIqny-GATde!DXs83|zB+tZSL-`<2Q5P3CRG(AxD#P?sls_Z12D_9yL'
            '(P=cMg3rE{HnqCiV@%k5Y<|40+-$2Imios&^`Sd{bt|KuJj&P`M&UQ~4bR++5589NS-Mx2%%i)GfJg^V-7YcRZ>(B7`ZZFzuVQ!b'
            'Tiv$#Pc6S4##P@}Vs=AK5iAraNeJCg=IB7>YcvyzzCKO(Ub3Yibd>uvuN7$5gl7+6d#tbO?bSdS$$cmpRqL%>Em8ccGdPV4zD2-G'
            'R-)6?c<uH21DBwcQ14S(-j++jZnkM&pe2enX)cSHE2^hENv8)e>>{0>bg}WEWDLPrKzU@lG$|SS+LEn#4l1Gpl%R7BMFiXNt&A8{'
            'iDD6x^{tu4-&Sl{<{TNiD<f22i62(y$8mj0Pha%zgre`1p^-ZcAu&7s?VQzahx!sOeBD5S>hPxtkM;#!N&!pKF9?vIZ?BKZj0tIX'
            'yn&(fx=P7Cj&}O#@|@kAeZqY7%lXCI^J{mq*3etaGOk75^@3s2C9?ka7VUUw&=I7fv<5(G8vA=$)h4w)CJ5_?21OTLgJO%JjiQQ1'
            '@6OA{zJdO-p-W@+xjoe}pTXhwt%KKVVz9G6cMsB6!)1F*wcOY>Jg-u{9DLEVqb?yNq19#91W^<RH2R^A=7G)-H`%n9*h`Nm&t<r`'
            '7-(gQo129i`#T6x$6M9pHVxFS>DoB$rKa|Ew$go1{nVh@U{atjmGr=}sQyg@MyW^N{j-KyS*%)9jM;w}{ToA|miq'
        ),
    },
    'google_ai': {
        'sha256': 'ce418f526ba99dfd8b64de03143bf068a1597b355592a29ff17e735c41db80ce',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}dw<(DmN5LkKLu8Op0(IYvYn)vjx(CRs;wm2_>!^Zb~>(?rX<SdiXt^6Wk>ONKl^(w00{6RD`~RN?EbdD-HA*BAaHPSZa9}l'
            'qjBIxaTJWbuq84towdDtFY(2jIKBye(Hn^78-E@|f!Go7W9~U3iI<t5h-sXN>^4ZnB8mUQA7`DdtzX{$Du!<d=i>PE@ZzX1`hOjq'
            '56`#SmH%!1<iij?k1>xYOZc3B=O-%>#hE{f<Cz#OgD?|8mWud364y6=<OQ9BFz}<S+39qgYvDzcty?ddjN^$v5!Wz`A9lBIXvXd5'
            'otN!t=%u&Y*CL3*Ao4q6c<W_i?oE6cRs{1!oTkADzMRFP32U20@%`5QZ7{wSG*$}J^TjmreYH$6_9Eepv!xe?E0Or#__igQBk1b;'
            '3*EY%1%9%1_WpQh2>*Zf{t(tOx$!f28PCPY5271*l0-1+(Mr&W<6r{o@<K87qBelIg^Lr-u>%i+5EdZ3twrKbf^p`-jN$ci63|a^'
            '@2zlUIUEC?_vuMrEJAOLK=!iztt}z;I${iy&iqt(VoabE&D4iySv!a({=$!N8ZDuIj#uy;7w%6S=>M#Pd%?pKga8CLL7Ks~eJ93o'
            'G!1T+Fv%PU;_u5i^I*8Cw+yoshIrl)lOP?(u-7R-bK#AGFo341xC=aS&6Dad_&Xtt-Q<l15CGo6Tr>LoT-*oQEsO{M0Ra6Ei!d0&'
            'roz6&_t*_yz37NVFrMLBCq8sWv`DjH%s4C-Fg?V1X!(7IHV|9F(-c|}6#D1G!EtZcKa|S|eb~hW+Q00;!m|Wty(2~*ej}bv;Ws>^'
            'QN}?sUcz)n0PPud`9nwCd0{Xi-~+^#fJ6u?n95?AiF5_KKhE$Z0%G+}4n^wEy+}TQ{y%g?7~iDwkN`T~0mdZ4%QAny$PhX)ZHt#='
            '!4yV<E`RKZ)QbSPAAQx&PZBt|BR+e02JiqQWMLwSUi)NA?Co|CsBucDlKKlTftj~7pkb1cH-~QH7{~%_4?<It4ZW+Yt*sMzngKbN'
            'X@-XzAlHVY7y5I=J}+7U=y>KMKnlcUe?@0A$xP}11m@@d+2Hj6cBA*EH#j*TiqqGx2M2@Rk$7`@`UWTt(5u7KgNx(-$*?yZoSq<V'
            '`{JzMJ3Q`-#yf8nc{4GIZ<1x`&Ao<GLz_PTzpb`7reiTUOc63ZpZMD>TcrCtJ3zl~2t=KUf43v)l(z_WCh<7kVVs8{3Wop)pO3(I'
            ';ea7tHkr-=32X`ZDAj+#3G5PaYykok2J+`4f1*dxB((O>5qb7u9GKn`cMG<g=ieq`EYf8HsOICbZf>~7;0BlCbl{^j*=r!jD?ms*'
            'd7%en4k;Xr2iOSIAX_c`wDawOY#NV{6uq*T#==VxM-K-F!*h|wh}3E3tx}0XNFlcbEpr}^XG}1Ga=@`8q&q)+2WJO*gmAb!2z(ek'
            'ngNA&LpU<wHPU8(0Ys;PE17#?L;OQDfXL9lZ~}tmoPOR0H@6MZgmcg)KsvYsn?Xf=5<Y)$dUn=7AmqJEJ2XYbdPYERru#5KIPi!N'
            'Kt8Tp@MpYm7wX51nIdjI{7cjZHhdXnyqzbfLpXkOIA;^aOdP-Wey{^z=!nzl6pl9VpqnN1jDT?fnSimR2=JU3OAJ)dy93O~X8}Zr'
            '*tDebz-4xAh6G<ph~&UvDW3h6IO)S_7U%tw!*eF*C{U;mbv`KmO9%McMNwelng;>#fgsH68u)Mk+r^xT0JWVG#{d$ewi^)zh$HY`'
            '514A0`pD%Hf9z#J9CbE>z1+EHo$pz;Xakk=uMsc^+K`ARd<dAeP}K0;_Y#=l4KN+p5%@YY&J7QYJZ~utLJMXbmtTYfgVW@}{Yg9V'
            'xh1o>Pg@j`2b^L1*)<$_pTCeU7kG{krI1;=cqFf15#ywTgNzK|mjY!_vw`uEC1w7HY;($>zn(h81NaNDKhxj?tQH1>t%?&A^yFt0'
            'r=w~39-ZFTTJSKXuV@K_gSerNz+yRqt-8&J!Wq2Mzr-R2)B#k7j{-|M&p_0QiG%>10Kp*)=7{b%TEklaKlGXXNw~I%%i#k!Ss+)e'
            '8q^nsFc`3b6C!aU6}Vd&o}PIyiGX}$;=MNx;1G%|nEQ>*frqD}&C>$t^^w&hO}T;f;l_bL-am5B`forWKl}As3W$6p;V>MWp1dBs'
            'xj5_n1U&Sx|GIZ^G=xQ<#6ZI763$No+#Fai9ov1eT!gVVNn09{_!H#_78X#RY{9<?5EBwU@`rHBlNh0xZjJox-uEMQPJqrLp-Oio'
            '7>U+QVvx9?I{~b-)o3)fh;F&=bcv+Lb(N^#MG=mRsJOKypN+iKfB8cFbqnV;3`XkD|48FV{f<-hD}k+w=lU;S{bEE{f8qA<M3BPb'
            'Uuqf?l9JTZ7PW-qo_YN9V;sqj3opajsDAI^&#kR*#FrFCz7cOhIRb13iESlPpo3%BDLP%LPv{FX3r0j!fh04rZWG%>Q38-1wu7)4'
            'hMF!TI_S{vhJ*ollg5PAVj2dE1)>|Q4w1_PS)LWEG+rjaN5;2abi;ZGi4@<U-T>I2;(#d#%04MpK<qnT7zSJ4?29u(4Ai-7jO!g7'
            'oL-y^&v)LRo&H-N<p;nxc_vFL3@nR&;D99E&+h;T$T`G-$T27Yth8M-qj96yl{oEEPdAYWioz8fz9q^nKxAORWEqKIio!5L5WxxD'
            'E9-~@gd5JBPVhNWI*|Ga%poXVfKf<~6g_VOw}!z&4;6IYZtM$-0tG0TPT|A@S%by?JUHy1IXLMlsKu;ZrNBvfca(~cETFU1J32c3'
            '#g(9P|8j9U?73&BQoinr*T_f-$~oyxm!o+=dz&ih4%SM#u7o^H3Gg0t7?KSFBAt^Zo_P|~ycG@z-9USyZnsO&G2pMez`sXA{xn%c'
            'bx+em98GfzQ+T`wvgV^2nvgC>uK?WxKw0QFg58>TwqBq0`|fbiKXVUzuy|k~gM&USy-C_<LkcZO>z7F5(%_?*c^OQ2A9x~g<s=9J'
            '=U~hvn8C0D$W}&=R;ZkXp4OWhJYWW@7VHa1V9<0Kh0ssj_$XTh$=nC7I|i;bTZSQY4@uEVa9F?%=y4N&1hriab>jo7`Z$CzMvZ67'
            'NcL+!3BVzId8)>{^=4rJgmf-|s;#_LP+YPnmpu1ZLDXq%IeY-Ud(j-ni&cV5&Yw7nE}#w@?|8&tQKBKCkeLhPixeqe0m_H8jbY3h'
            'GJ+MQ*bE4EItHy9<vbwW_o$JlASjS(KKC;ZXt77cE=3(j;`3SW7x%b-IOw^<Q`gvFW_~$dp97*_oV!2wjxPG=@c4n@09r77e@+&g'
            'MoTm%NHgOFK5)t8avO&*Lwv<Z>n8r+m#`I+MvF%PxdG9Cf=yuOmLz&(cbqOhM6(aSmS+QwlZ)qJ1w%Z-%nsi6k9#fv?3_>H-qzOE'
            '7-Z5^qEm11o(O**RsQA~kRg~b{o(BM92$)?FTjI&e+ww%iQ5vlz)iUeVx%aO4l>`ZE!Yw9G=UTCKZ_F}Q~FD<VL2)6-f{w)hop`>'
            '?tpkCR5x+&v7JJIb~z$_jJh9?awa5o0yPIM2G{-ve++cWW%PB4XaQv<t`x<&Eg}qww*WOk!ygTZ)-*CN($Wz%KvhSgnz2z%PYCq_'
            'xd84s73=6BuNMDH5U`aSp_|aS>qhGilcf)2Fi7x3PrZ+!r&a|F91bu`N@*ZZrydkedWt6<h^coA=N>3|Gy$rU0Pz6nb4QBCEXN{w'
            '0rKUgb8jYQ$mt5B1-BvEt(eG+mdtH{39Y=;n*mAc0DmQZvGT$w_&D}N@7I$7HzQg#@ry$eH-jU5Lp04NQq;-Y0MYp3lF&KHpCJ7F'
            '?1f98h4=deP9bbJi7|kOW0D~eQ>0C-g<9zxO>7gCTwxd#8rxd1W%I4H-F^+prT+mr)&U&Akz4^;-B`^9o(l4LNHhWDQzUsHg@?<O'
            'ERy$dnA>Ae!~m1g27)Xw3_u`;Mgj8DG!B=H<bYpEXY0IwaMmBXy`w?z+@y8#d;w2r?2D#kcke)}0n)^FHy3!H`>y|M!;!>o1qiVm'
            '`9#Zf$6xs&wu7F(o?Z+ue(EzzE%pa>_V4_$v(vxmw=MO0jAR7VZxH_#{Rt-2{^02J;_&rR@2s!S)ZnnXBPca)7SH^s)bV2Y7P!K@'
            '{z=}^`-r+my72w+t&18R_UZlY0-B${@AnVh+7q;zqlG+<$~{mWJLPWxIhUNrg*`azp9}}RBNqg<*MlRJG#Zi(Hn!A5_oR2+hauo)'
            ';&*VHfm<|_#=l?o+JE!fA9ve-yvl#No%YqkZtLaq$N$rS!*y_a)ceVm`~9|m^uBJh^@(gS3IfW2-V*u1+h&2LE=_s19JKA+06juN'
            '8q9!N0TbB%PW<WveYpufh9;qE$llx7s5jXc+ZwU9Mf1*IdBRWeBNKRIVcE9R5{LecpX{5%<pN<!>bjC@05m(W8e}uxB*DocG4;Yp'
            'aD!ZcC($uS+EylO0mmFx6l|+zB<39)_kIOQFIs}Q1Lqy&o&}t%RqD}sM2zp;_!)3OV@L)NjQbkRG{gsX)K_}rG3Y_`3}sGD2-lsZ'
            ';=afY(4fqZS_zg_=MEui%vJ&yHNY4lh=;}iCw>J!FoX|?LzJyh^1XYmf%GIW#~=pB;T)irFZEFiF9#qbp<?BQXZ}=+4?QMWDDkl+'
            '0T_1-9QF>Wx#I7vWGpjfy3m(uAabO)@qHxaiR;8)Bp@p6T!U6m|1z0py$odl5;uv+stXip`)Ns2jxTM|EESN_ApY(A^hEOpzNAZ0'
            '^&0CHDGn3*-i&W-(7YlQlNK2b1>W)qysac#30-ZW{TT^05dPF7H5!DSn_DE@LF6SX#ic;m%f%J?;ks!UXQ}I=y3&{d@+0jfKHws>'
            '^k07J;&qmiH>W^N@TW`gMVf)5q3Gwja0)WlZKi$*`YsN+kH(Rf$Va6I;vXacwuCi)i+8ML$piGC{3`eDih6qJ<Wd&Sp@UO^4)~A#'
            'BA*bv&i|-rq-F>0GSEK!ruEdZd&r4GTXmo3<~!`lw@vwPp)bReyTZS)!3ZOU?*9i}K&!}B8rJ!RDnXlsYB)=wA!ahR;=`u#0jF+6'
            'dYccHZ&N-b;0{_}6HwXsq;<q89)@dagJ%Z8H%541;9xC5jwVeV_2-U~m5uosQ_C@l&XYBl(z+Ts1R5xN(>TU8vyVpn2H6^-n?^o#'
            '7Q1Aa9J_d%lz#*NvQ~PoPIpTm!*=RT(J;nJCE0flVjv=*q=}PLMSaljf(N~7wGXziOQ&u>$5k8m8w3XRo?+q?ny3T0Zy{izk#VT^'
            'i}+vF6tG*G_NfP5`jt<FF_lM?2p|QOL_qJJ5Z}Nl_HO3hJ}O{z#V^`;C4L8xDnh9_ZT!ii^>l)}4;oI7j3M%ggv?_>mB9a|3P7@{'
            '{Ogg=ynL7g<MGdmo&j~t5~7W2LFN*fsA@_F(%kqnmrCr0Kqd8OSIToPW0MO41E5nAMSG%ulEf)qly>~+E=T|sQL?~RYM>1$5D#db'
            'Fu>WLwJU<hHvBQ#c~P|?^j0(ugbP1pzDgVsFX`r9v;<ZL>+#7bp^t!~ZOByB0MNeCaA5b~Ff|>eLn99?O0u2lG6bO@8{amWh82}*'
            'QrxO^XoJ71&+fHfz5vNA0R<H{hJ_7OU$u&Q>m{(K@ZOuZ#CB&J{>R-GBr0vUx5-#H_rt){NsS$ml#I4O$|VZya!Z>AV>CBH|CR0&'
            'HL75Q`F|F6bdgNcK2}ReXFQJ-<OQH&N_yeRfM^b)Rd9p04G=oe(!slkf)VW_-eB>>q>Mz@x7y~<0u^5^BF?}D-tvVPfg5EaZX)k?'
            'gf~UN%X=FdkUUR1;##dAFa8&)L_QngQZ>mmS9zr{M7pZv;l!%Z6jY(^Dyaj+!Z2nNdL}O`&H>-@#hTU~4T?`-?Gl?owx#=$v3?C|'
            'Hg2$k6W0S%(<sLQz5Cvshqhc2D^jA#O*}?3hmITyi$V{KBR)JjPW}0qHqfLJ&HWYVI_2ZBFCNs4{v~;Q1#;!xc$vXdJ&VT%k8beV'
            'uAcR!>U>3Q6r{Uo;#JLC4noF&it&mG67g=9w(@rtOx3`&^Q&e7%$^(v-cn;lV+dd#nfSQ>IzlpSn2?Yw4A{bsYT`$<zKI(j$N@Z-'
            '=;%$*9-sI#pbtD@KB2gE=846rRa1KEMDxTc8gtF3nkB1!i#90gDtQHU$Wefm1kqQzLUmsfHr2yGii37jN#QISL&Qu9@~zy8#FD&1'
            'i>?BsMJcPsvD5<fXL%Hxyq3x8=TS$xI*V`6txoNyQm5!npEhKC3*w$q7Rx-^p&NAN7b%*ogkPKbKl<a?eEXU#h;(2CP{>S7Vjz9K'
            '!`8?AN{Z<y-wLPVTI=2LiEEr93NxZ73`8QTMyQ=KQbl4JrGB;(#^YJ8Fyo~^`F@gvN&^qg6wMJ0>C5Y7`Ek~_|K!V<c<HT8Df2UL'
            'TMbD-!9A)v$om@-EVV7mWC=70N5iw(9KnI}BR@jw#T3&1w)5!x?{WN`!>5j#mSD!9HBMaojo98)KPtqTl3dxFOvu|qTTK*WryH&h'
            '=&gl5dw9v7+!0s4rK@ajtnd2BGotf5d2-%uohhNDaH^%!bDRf!7CkdG3@rEp_n^6^nLmH*XioAdpVEpS)twn@lp|~QtkmQW_U$bz'
            'L3i8W$4zZ7b+*Nh*p^zb@w_E$!zoMwj?W};wk2shp))>K+AVGcAyMhhE1#jEeV2jPFsJumbTX7Lr(r`-Px}F7@Xmk4LDVev)XM0i'
            'UV?(E0nA%JT)60SmNN+(fa+8ft>2nxwTkvsvURh?I0$c0GEhcNfyAiO_!>7=9O1xz`Y3xhD}3=^q1JgDOmJpwiXpN!TB7kiO_{?v'
            'U;IYOyk5>R^_aJ?>WUeibLCD|nU88Qza>}YqD3Z7R?Ob@Irs_3LONr4Bg=RO@~*<z2Lp|yPUFoC{Rkz2YwN4mXwR37j|?K34T?n^'
            'hcbMGo#&W1NjH8E;~=~gO|(G3g0{SslAc5QHaWjv^gkd8ls+-Y{b@KU0kF^OQ+MpwjuKF8P&K_pfLB!P1#Sr{=<EoRM=Ec~Vag%;'
            'Kkr0-b{{7*A*T^UcVuZVk%58)p2N3Qpj$;wKU^&;HTMdffEmt6=(cS`#QYP{#Eb6*XzzE+kINkdMY;mO0&UGmI+*((ola3{motBg'
            'h#5EbW!&Df4Nl5|?EN682&;J35f7CF$@nx0M~@X#w0BY6DcH7INx^D1-1v$i3(#?35G!9pJ}%-L`8Qxj=zR>(77$AWiF{I45j#xM'
            'X@y|be^wf~l<2MwJN?sT0i@!1w~QS!&%1`r1~4fzEOINdJ8ZtujT2ve8;FH5C|=Zsli-!S)>Jp4=vX!P(8UTJ!RO;Z%$6K$^08B6'
            'N?K1AOMtTf{2J`%*b3_c76Q+%lg0dc=Q^YRO%@XlCa7{7J|{Vvq5!2=;&#dbbm$qVd`Xv4fZ<;h?KVcIjB}6%(R$>ki#Vc?qX>`^'
            '8X5lA((C#jG&9hD*omHEz`hYMrS2}d7BA-T1M!62h{E88tU*#O@|IaV_jCl<7w3;X^0Rk`{lB6MS)&-Z(@+i0hi4ZD!wZl{u~{Ar'
            '=5jEYZujQw^y1`@TS*jQ8=R}733S`x5PdcpS(fTY{o|jwuNC;CU-#aCG>osk8^pG;tAYoVaD^TW<L0+540uFm&KAPAB{KXU0C`K;'
            'p)zT=>9k-+=Riuj4R$4Lm<CSMYpzQUc7zaq+HEx8Ba5J02AORNe=z(9cutLdb!EqY8y6OaIO@X0p(Reg@^wb9v%1%?Sn6$Dq%;a^'
            '*YhYmH|N8kftLKkcw_k!?(=<~`OF-JAlBGt$QjSsHvs<Ec*Y_FJ6pobhCGw5u{Pye!oBbw-i{W~z9Dp0L~mbUpi}$rOAiD5q?mf='
            'O92Q)`KOMF^h~vbbYXUkZn;U6H?-R{PE|{kPLO`24vPURBnRIU!gim1zw7Sq?h@j5pY7&tN;s_JyjmRQ)zVR3Eg9q`zo5O9)Kb3x'
            '@4gh?;-BOegA}-&`(~^vNr{_K!b-SMtkhQn&R!LCzTf>iJH7WMR`g%L0N&q&eR-49R$*g8hmEOl|7V@=jfgcOBaW!v-)H}Xc)u1H'
            '_@87ubs>6m0U!f~qd6OAF=nqKB(Kw`#F1xT2J4GBvir3_Q4o9&FaFj)8Xoue5jEyPDSDA29m$za{CfJXg?_=Sr4f0c2T~~J8j*<^'
            'x+i;RghXjMFh%1EbUoY`DBCm_>`9L%9lU!S2eT{?1@C%CY(kIJ=>=J-El1s8gNElVha{lxbKZnKV-NO>J+Q<a^x>|NZCr|bXq1pr'
            'EtDYrN-h+a{L#y!VDyro#6LM4uJ(V-Dj72XJM*Z~&y(aWm>B`EPeY1?YXz&lU3oB{o6w@4`2YwVTM~cb5bAha(5}vh{dY(GljHv1'
            'Bo#8^WkC&%0bT-w#}ahFRS~#_vpDy&2%U4cao~N?>2$<D@PD__7B&l3Av#EWEIQ*?0G?98h-SQi^DYCmV-BRvgP|O;8Q|w$`$hZt'
            '-l!d98&XMypAzJIFAb2ta9`kWLkXFCW!zJGrkjp${kdn^k6w`<r-EQW=~Q5|IQ+p|B^0v2Fe)89TFbMSqqg_rzx{f6r(c6c0lMK$'
            ';@zRNZz&}$5ic-p47mo?Z?|-b&qvV|Y4`i~v)yMep9T|YstQgo6`Xc0%4D201uf&7jaCK0DuNL&7gi~7H}5|@iP*s_I1-BhO%+U`'
            'ZoCACg)>Hc@6Jn7Zxw))Al~)f_l^dYQ2d*p#BDYP(>V!JE2+@FG`FB<iw;IX3rn7CMGl7l_5XTsbb&^JrgBC0Ka9gAh0<c|t;!4J'
            'w6sso62rGdyIy{SiF`HbO>URmsNE9OL3GeTgetaZmm-l~iJ0v*GZGqBp6Hz)^~4Q^L4K^^d&u2kluzL6b%NLW$yczy&sfisOJNoc'
            'GX9PKY1HH?r~}7XM%j{sQ9;1_a<eAz|96^kE-L-A&-#&q!iKkgMZT*}T^b*mU<ydX;AGf8JMSN0LMyhM?h`S3B|ItCv(F&eVpZ~|'
            '5hcp;wDq0hCAEbW?{quTVnnfwM}zaBcB8i~N$O(Y)=S+ejy?iW<U1`+XygXQrETu!j#~i(+C$Nmt?Fgn@?i7Q<p2lsnhxpE>au7b'
            'yb^zgt)Mg?<<}E_Ete3auC^s}<f*G{&1)A!kt_^q42;K=N>zqwcD6+aIs&`iV_*-M8SGXjm_Z~XFs@0xrX4Bd%GK=fGO%Fi!<E<X'
            '&DXeV7-EVST5&3x@8R3Nc<yw>DLQ$oEOt{^Jvw-@>nEaWx}=b)=`s}4AW1Wd@x>6&;pzMHJb?jck(qnQSH&V&;7z`AubO&MiU;G1'
            '-v9s$9QNHF`kM5HgP$o)K-t#WsxTV1>I`PBl9=qs&-~>6baC{~{zOIB3X@f<Qe1Ta8YqypcYkQ_{eaiiA*d2T7-!3UQ6!c!-C|B>'
            'WT@|()3b|{i(gOAU@*|+V3eCA5HEILU=;Fl6(@Mf-iTs>ez}PK+VeGwVR!B+j4thn_o0vYuw0<a4~(^nm*}f9p?1(lDjqu};9|;x'
            '8NY?;<5e<}QxI1984Tsm(RXvj;x9!4BJO*_OJTyRKxFU_aCQ-glVyTaCIxF5Et7CLgZ{K6nRy`uay`KTDV#a%h@Z42B)6_n3Pew='
            'fY`%_J97ID%=?`Uu|#eCp}g;DuD5Emp0Ed~;rjZns9B{9w^X~=`qHSmR-%UvTf^S@JD0i{!Nj+5HjgPyf|k#UM@BMdj=xbkzc_-n'
            'rYS}$l2S8XJvpqrw;-!TR^M#23}-I6yI?vd`}=(CWW&24@qj!cAqOC2Yg(#Y6je*I8s;f2U<SSyEf>}8N~Sg4EXOQ0B-wrp)RE6~'
            'BZ7V{QK3~*+sdDj*r4Q+_kPlDqG_SUC(Y5;NI}-fj_8dr2Z!kX8GQ?EQzepnPS8PkGkiNZc{ezD;~w>Y#>6-45a^-mGRZJ_R@~3%'
            'xAox{wbvJ9QGHy4rC(=08&0V5EPSOIZ8)Xkk^2g>*=!=!hx(zuvvIvrclD#&Jgz4QU-&E4#N-Qqq>>PQ;crwCv1$a77a3L9Avam7'
            'M<`T+)Qy5*o{5(K#F+6&3E-HIR;O~7v5=Ddob~^5F*xfVx~%c7B|oN8SWgtnttwAyxmoe_lv`Azb-7&)W|kXPBUiayC4Q8fRc&#>'
            'WMr`^<0_(pOIhakwM@z~9u|;8`I@tG+@?ezV(eoO!i~&bN_p%+mhJu0`&Ime^W?;i9wywDdE3%kN`;EOIRi9eS%iV3A-Rw5qrFes'
            '#3^@?Vu&&yI5NG3a$03|98(U=eUPrb57Hfo)0CvwtOug&rA#c4QOZcban9CB>Mzb`9!LI4poX=N8e|5d>zJbQaCEW;&x1*u6Bx88'
            'zj&&g>tU-o#lbjEDVgJW|KRlGkaE=de_#4&XP}g7GKfK01M+l|R!Tuapm>R?0x24Rb5wJZI=rdK%rth&iY?<&R0dE?XcJJ<Z(Eq4'
            'tWZg+vc;uY5QgM%Y~=$l!?7#ViBrO`R37-_-d|miY2IH9IrYoA;r2r5jYO~L?$l+Tv@b)7Kv2erVT>JGpvKw!&%N(=cg64wy(T89'
            'a<2VGP_Ple=J`w_j>BtAm!Oa(s~t*Ghf0;o=2L+<omy(N_tJg-!;3Y9$ZAkYgRB-s@?*YKp!ul`8m97iK=S07WS&@fD@q+oCXUWJ'
            '0``Nu_s3`JNtfBAnrN9VD#?}6KrQEBKUdwL2PGWDl+wHyP7@2L#koe=OEHLas<}=*(UaeLlt!cyW@5;Blket~=?L@C{=?3shE@&3'
            '-Xn>Y40}X(W5F`GOfJF6-Slxn+ffv&Fn6MXO`x>Kbs`WGA|*Gmj*P2Kz+AL0Ri7oY*5TLDhNPl#u$7EYk><nUuz&oXwAvRELN->}'
            ';Ax4np{<zDf&%|=Vtj}UsM#a8$jYgVZe&4HWerkHC2o4?<Mogd8dPd<pe0Sd6J_w`$CW>csq3Ia4lY)Cx}r+g(a>sPlyAMIlg22S'
            'I(1Ch26j^pBsTX?s$x_1gDs`htK{(55=}>j@p2NAFCPh$OK+-V1lF0Ddr>e&-XTqf&%S)1OOB)`gV+7@p^MRCn5e+;#LZ<(SIww{'
            '>ERmX-oxRM`}XwW3~~F}i;l7)yVJlACrx0)EsAk(sUukkcR38kS({Cr)C0IpJX<d4$YcI`=H#hLO%x<%fN3hjL~s!CdT@3=RF-^T'
            '@j$eK@xGONUuTp<RK8}H{`7k1x&s3IoPNywReH@~pjnm#BQ)QVL~?zDq&Z8P`aYUc9EfcmWZSt0UWmzJQjDg{sDKKg9UHSpNHLL)'
            '-o_J^(_7Vs(Bw(n`RMvyicJ9+RJi){@Di%3sS(0Lpr<hXQYUz&({1RDaqMUaUTt~*bO;2Cf*|_1eCQ~iGV$(9k$pvDY9h%NXZs6-'
            'nJ>0$44&JXs!*PUu0kvL45&-^tieeLaAV-p);&ljBbqxY$V+g@cN)%Rd+$mo?BI3atXyrVRnZDOA3Lt&+YLiI>}))+fP>!2>B-=r'
            'cO(u@kKy1P?sGn3l5A^kw)l#Qh044V(T|Mpits;d$1@Bq<i6Yc;xd?|SMY!mQf#XqE(YdoKX%}pC`#jn$Ubob94(27<0)max-Da3'
            '&o?l1pU+qP?Y6!!yT6r+C&{X$a>@`ZS<8m*;W(SUy5tefLaL=Qozbyh2XM-kbCUf<F8A;}_RAJ}6zFwIT%41AjI1DMoo}sM6UkHD'
            'X>*eq$z&9CO{fh9mqUxv!Fy=LGZ2<!d_aTO01qx<(;Bo_4dVDs=<KLRuFSGzB!h8PUZ}_N;uJff<;iIBhw9Th=Zq3_oir08<1D~F'
            'IyFmRw}5;F`B#mV@cFP;fzKYdsshPtfo6Mp)k4s2rlBdrwH3<IbA+$8r&XQRjHUFxDve+#kgA9TEo1j*QIxfIO5?wL4<!AP!-0;B'
            'Qf<UJja+1)9EYKDL2vqj!XPe6a_EO!*2P-Ynvb*Tm!kN7b7DH;kXbvWp<y1=B8-w26cuD8#H3c&h3o!n`hF5G#K+vvNxB8L|19&!'
            '%onIyQFRbMX=ld8FVRZ$cG6<?GDS9{lQouih8%9LaU@i^k*XaJP6o%Y-gFL{6eoT5ZZN7n6Q?Ifzv_tHMG~NrEeSXX7g)hf6*Hll'
            'MCHvh!h&)B_qTD#gTaOYkGN539&=!hZ~gI1hx_9Bq|zeZYCFXQ%(A`-%4jsKni)oMV-OqFIpMtmS<9x;u<3x2_`lFnm6_<k-%V9s'
            'qO;7#T2UyWds#@zMeFV+jQ)v`#7&XcIx_iDF0l|edDT-jX>9d3bHcYdg)hPDvF<o-K)|<3tcDndUd5%F;gut^s^oqiU2!?Dnig*A'
            'yzIPjrMN}7M56=XoxAyHGAq^E5p$1_!eHiSq-I*-MB?ype6rV%P~BTvXa-XQ2{=W*xJMyPRztd@nsSW5bQam9s#QkgY{?OE5UODX'
            'x(E82aQ19zsnm9WZ7xEo14^+*2%LH(VWj4JksOZAkRzWH&hCI-6=Bx3csk{rt{}4C+;9q>WrShGAgs$sd%ZAMJqJKhD4>Bcfb>_E'
            '8KB%NKQEQwE;?(i2RUP?JHlK-c<LT?vK-TsNTn<TeaYBONBjc;)#Pb7X5!W=<HV&J@k+Ndzn;I=w%I(1rpjdL=3$|n(ULs5)WfTy'
            '3{#2_t#%Bn@0C^e%GUSdc+zSwvqG;z1Sj{g9u|9)15*u16knDgqGMFKz5KHzINI^5+}3(n>`~_~2fa6OrYlY}Ct)yviHU}cZTC=N'
            'vpiS+>#CI9RLUe4qO&cI9rFTd<7B@Uwuv+?;{nI3DBE7)BJmQ(x-xSDQ)Uj3AZ7uTBIuHubD1|Oo-~?9F-H`o9zi{gfv4QTU4Em1'
            '*wWK$Nz%}&ARmr3craDtmjs96DmqwAA(|AdmD*w2;^ew7uUmC}V2)_)DOFIx?_&Tq{V>%n+ARiZnJ6$c<h50v<dz;6@rm0qP&j`s'
            'Z%l#A@NBUG|0<3m!7HgyV?PIYIqB~bu@ULHm9L6Z8b`653riu55}s;?X0E!y;UBn@DSzHB5@Q60tNO7x1(x!RZy2Y`^)d=IzwZQT'
            'tg@t=QSe#l-U1!MwbYwED~8VT%}|W<hm}2)#AUEUesopeTuEfbKK$J5n2Vm6V&BMWQphG<%4}Ji2B%IGragX8iJ{G!3#~jKbYmG|'
            'IqIlkN}vtRT@^MLxVclki&Z8FGy6cb*a+?GURCwZ#Rs!v{?zDWBDW!+OF_YCs9TtlfQf$?BFAI^PaFNPb2ym#oRZefED}|@l@R;@'
            'DXKwao(nYcx#(NX)fl)^7+PAYDufyR^5bq7jw-frHK{8^0<QebkI<`C9{WJ7U=KsXX!sq)DTP?{H3>otS+1((Bg2*EAr`YhpHwVC'
            'mb{X#zJ**XRCsLU-GtC%j0J$oQGt>gA(N=EpX6ekoc3jDLy&A^S7i;dU)xaFO#iIbCT;qw=F?R_T7u0qmHL0HI&+WetJ58A%6uhv'
            'Sj&T}ZZlC=Q<6ZX%2v*wukiy&S1PkvZz6jW*SBV?zE}{bE0O>zTwTR8(R?Tjqw@dNYl=#EC@-cG!FNTo7&gOk<sb7e6`{s(FMq(Y'
            '>T<jv*U?%@E?uFaF~6SqDZZ8o13Zjsx9-K|1u5V9GupTy_5wXgZ=33g%m}XSdVrLQyY7SD7<<YUK&B2Y#Fj{qYRNz*M%IQsy2Q0P'
            'yR#-z2EL7PJZc7XF`oaCXX0WA*k8$|g4P<Tz<I(hudQ3?9F>b7IWe&pbLDKUPrX`NqGccJWFLcQ8rR#$cusly#HUJsP43q%o8Lag'
            'RHup}VC%N=6-WH3o7a$S5>UH@3rzsBqc7-UZ{mVb!>08rn>t|HGR#t-1Bs%b>kaH%pKiS4@nD<AD0UNaf1V9}18aPKN;VyH{bBly'
            '^}1Hcn)1=KP4eCt>rLlpJ@Kjj>$TjNohQ5su4U@nK$jdsTRf_x!y{>cPJApQ8&Bb9q^d5B{V>eE0~0I~oo4b%l8o2-@_4hdHe2;z'
            'KrB~R;-*Y&)Dv1$)MqQ_uV&s|18p6#Q=6i=)?Y)XE4FNA7K)PPk23ZmMaD%sYw|vic^|t}39v)1Y)$HnP7|E%(T~P4x+-)V%WT^I'
            'q2V|cBe*%dn@xViw9`;Sq=OFo0BJBfPhcwac?!;ka^EaYr)ySqsQgr!gYg)ACgJ}L#?{roO!J=&3Xlr%Uy=nIw?S5}@S<`&Qu7Uy'
            'Y?CLb_(qK5Bv~#p?u^GW@rg<n)#xX0CAYq;99ng)Rq-qbs&PULc{+z2UUxI-sH(Ax2nu`=Sm&59w57YP6Jys1<6q}mR?Fg2<f*?d'
            'b;?xrnEgz3CJMg(qnowj_1~JSKXvCR{UL$2p1yu9n%8F+C(1?f;Pm9AuRPn&b%Ee(%Bf2}-%N0w4rkRiuk_?S-YY6?yW&D9<oT`c'
            'VeK0}CV94DC-I`(pCBW1mTNmkm<|V$Xclh#$>xbVs4jw&`s08@B!dWk$(TuBx{vu2PH~zo=UChsnFXicM!)Bd&KLO4<T%1$QhpEh'
            'aH<hTY`}f4vx#LlCj3B@-nWgGMD|TlU6Xm!qLF)mlJ~6D&azlKW*Gr#sjf2Ds&AY3L8d)BcRfYl&ftkacjwCdED0D@)@g+4rH1WQ'
            'ZtKM_Vctyoptk(V`_y$CRTqw?Hmp&oj?PCOwYW#8a&ma9ri*?Tyg9NUTKe{l6zY;)UhL)W)mCvd8}Y8{?cycrLWe+fOTI?Myjsl?'
            'RH$J$w>QX!m}?6$vkXSQdg?wjN}lNGto~?sa5mi4@~NucX2Zb0<|02&Z4?1o%oo{8rAKx38xp$IUyf7st+K{^tgq2gA}bpq$d?C1'
            '9dY_S&RU=6uw?*DSF*!Zeg#)=u77KLLW%FxT4f;wcK80nmv{H>pPikaxo5p$-#r=}4~Fj9`(qTT=)93Fl<tA6+8vhKO;o#|21iHe'
            ')=yR4u^A8}4AnqIo$bx)?w17|*$wMYQ&#3>cB`J9oPtV&A!sD%vFl0<43z5k$TsKwv!DBClu(*7xX9}g`F1!s?gNdd6w$%lk1@-%'
            '?ALqUAO1@Hys1wibW>m}p?W_!`MGyAIFunB*e-}jMRY3?S92Fb?>HWup92uETLrh2?JrK=ot*xHVV3&VgUg@e3Z#b^S5(MiEc7Nr'
            'aFX#d$ixUVubH20r&O`nA1^T-aa*}{P&X`ZK0NE4oDX1DE`Rgg;N*}pD45V|2@6_|ISVdn2U_^}Hcv8wyamfjNq3P5!*6JRI$N)M'
            'M@K*P4&D{NN9U;u<62cu57kqR0xe^J%2}&F?Tu?~!D>|5lcT}AzWW{{Zq82fNm-S|dxMYO<!<|rTtwXct$p>dx7&JD?ul00=8u8%'
            '+XyQHTw=o?LE8NXqHmnYv-^*c?<Icnj}L7mH*ILxhBnaWxAfb~{q7a~^V|7%wPQCgfBe_0=8yX{?jmsc1C#}Z8J5#^F8}>se@lOB'
            'Y+rr1W{T;oZB#`!*W~B|)aM$jPYJ2Ha7A9MDv!%J!%6}Z4zTia-Yt!zbVexLvXxt4sK6zAVbWbKg=xv)qI4ByXeA$$6C`)XxTN#`'
            'aSsmD0pQcGpxeb*Er(cP?rlIG|1FGWZO79#h`~292ne)a5T>vcNC!A<aC3vfY^nkZ6-fd07?nKA;Ee%KGLa(%p)w{@HE9JLA`xdy'
            'As8|hWnW~J+FX_rn92g8f(eX_u1i5x>LbO<)V$QCw62&UKTU;xYWY<$l;Rl6N?LWZo3fBL@feJ3h;AG_v6v_kjxdH(q?mh&8z6&}'
            '8sRT-GSjIoW%cbHzN*`yR5a-_0VRgxae)*m{)K6?;_OyeSjh`u${-yhJcsOfF(&AuVjQX_7W68`xEPnt#O}+RQ8wtCG`@D6DvYli'
            '%BCuHL)NHnb%T}}#+LLOp2#9=WLBi18~CsxOP85OM}8}BQq4xJqatYr1|u3=WGZ*U!E&`;lCc*k{-SD|mK0=vLSlj%zEwGc{H<0W'
            '^Ls=D8GgyI>4fpUyko$bcIeNhqwiM<5;(6HyL&D1V)r>My|F;_P3gW7ARSPh>`0G=EtU~dVk+=}wg0U5zikx^hgd`aZ525ID~%`1'
            'zPM5dFyGf~E($c(GP+&L2Zf{MQcUqX_g*TD8ii|TEh%nZ&-&>}X?9wg*|R^=n5ciDf1T@pxb>E4RsmXQ`ozT88=}|`#YMhiubwE('
            '#>Ei7)WyIrSBZ^$K`L82m30Cu1kburCm<lLHXqX#O17SEb6mn)%w;dW-+hMv^Bn)@1>-P<7eqiQSzuicQ9x(qo?e10UYVgqzGAE3'
            'iyI&Xi&JXiT2L+yM)z@|>dnOWCg|sjZUDYN*$5M*)TY16OtLieh^u*)o&(do4Mss`q?KL|L16bXczqLy<c!05E+uy5`>-Cbsxl#}'
            '$j@lioO{U(^Zn(M<}%~zY1439^YJDXv-#pr`Ad5ErS5_pb5@)shOH&)ooa1(N8vo>DZ~!TC|u5HxP|ih(bua_p2pf-kJM_%((?rS'
            'QmaVu1Zc}WP_5gQnFZ2E2XneFR^_<iNiHBs4wS+VFJk@Z-U~>0@C3*+Xyrqx2Su|GvwcLED#THeCZ{n;d_dzB%7s|Mh2@SpT0v0z'
            '9eX6SQF%@j4k-;F3Qa5}DiP6)is$HKauLA<BIHXC%k`zNzD(mNwIkc&{aLtxNtf}Xaz`khihrV9hgnU`M@m*UalntrKi(K%;epUi'
            '*)#L@l8npptYWuOEU?2^WV{#K(nXsdHTE03jqSXXddUWZQWK4JARwf8O9}9HN%_R8wJ`|JEN4{5iocB`k}_o)*S18N3C6=Ft9Fl+'
            '*d5-*LqGIZtiDi1)<{Kv9U;17(t>6nORhU8K))Rh--~Nwplfjp1P|+ad3ceH6P@6gSS3t<pb#of&JY-94xdBYKgC@H5$99D!>szD'
            '9ryWO+7Q#kyO~q{w@Alr$OF*PVZL^%3z@?<_1Nj>7PI<TD97_o4cXF`;gm9$OK}a^rC#zZG^h}bWs6eCIn{UvsS6RjVZfKE8~*93'
            'KS6sx`?cMB4VrtSz^~z3auE5Q@`Y>3;9Ma3bMHfQw}T1i$;CvDa|y#<Ipuq2X+dr3@{TQ0Qj?4SQAJQ)ZyA8ci#Cu)FPgEyDkgbf'
            'R0i;u32$`;0xjhQ2q*i=_rTt>oKVPZF6>PHS!mypt?8Ns2dVmIxk4fP$kWNiJF#)>qgbiVvJS~NhElC-hciR?xvjhxM(YCJ9dtP9'
            'Of!*?{gFWXTjNK=;L`Z9`P<~X=8yZob>Ls;M<@NUVQ4Ej%0Oc=4PEy3BQ&9R9n|U<%{|957^~C^l8;ubeZS(IQ0fApJ()yAr>r*?'
            'Fcn8~>PAA|Y&xTAAkj~N(Z2&Y3b24DDFW;N07$GYD#KzUc;pD6lVnhS2Kl^5G}dB3#kEltOyn3&>B?iH16#}|Q}$%s=psU1>*Ebm'
            'h_%}>p-~rP|J{<^W=M1#gn-;~vqMTdbU|AUWnVIfZd1+$w&!$I(uWBaloU<Rk(CiJnU<0FN7k0-<YkyP1i6NnlIj>@+=2FbX^xQR'
            't%!H9gf&N(>y<b<J$-l1DLFhOxI31mQ_|j@e45f@<>KrJxE0Wd+%Bc_wj?Sqb(+NqWu02xV?wo8oRf&MTj(_Ig`5p+)__5)%rGRC'
            'l7r^7@qm?N=xBFb3Ve0l$9?et4<4&5O!Z+6oF8s|HC;I^GG!=7$zq{c!2WN){r0SX*u#wU@Ux*}e(PsK3$nkz=|}56$5x*Vbx`C*'
            'l_v7iH)MxSx*Rn_Z$6rU*u5{BbA9T?cjeC!?cf$^Q|?Q#(b~&y{CKrBi`L9JTx<f2FE01x@w@s8H+S#7G2Ym<QCUIVjMMm_qYc(6'
            'TB6{F(&zg_R%#G1QfT%knEzYe>b?B(w|c~J7Cb#So=)NG?26WuN%NN3cwal_s&s3b9$lQPwTX-2f#3o5Mf(q8vP6<24$p@z(f&ci'
            'nEdcQNR_U3fqtFn-b?<(ER%i|lye0$uqg9IixGgtRkRpV+2@1`fWyd~5t2)~`QjO1{5gV^#JgjvqqeuZzq?CQlb26>X=XG!iSICJ'
            'QX+Q$DBC$qIR^7Z=*w-%3(Uy6GlJCOw$wk-Obt`Et>k!9E`AB3NqnEmyiFgp5&Js(hzq@@{7snjX0j&543(zVfRe?j{|0{#;BVcE'
            '0T0NY?^?%8l?!!KwQn|hK%j%Mw{sMyZZEp=LqBae4*FR05KW7T%VBFpEO~waBbwZ_lL6L-T6S^7&W?Eg5*XPNZs%I!U4ckuF9V-Q'
            '!7U+~1d*|cPg~-7OYE7VgPZc!c4g!547YH2o_3orMVsGs?tOm-&(MeG|B63A(?2MWbcqwzm-hDbrzpu-{w3YM`eW*K;d?yP1P98{'
            'nx1>|H?5^~`1AI&c^sjuqWuv5e+^y4qKH_ArOpo@XMz?9V;<%@<ut5Eau(LJb(8qx`WZ-E0@8sY2O2PcGLb?2Nl%pMLi-1824D&X'
            '-$%oHCGo28KDR_GIM~}VJr<=QL2VvMvb-G^BOeo(;x%>)v#Ky9(A1kzR2q@miYO9|(C*{LLb}43<NDZ#T)oH-MO_?*^#!B-lW`bz'
            '9StH<?i9vjFSOcQhk%`%pW-eyE2bekbnqNAoL25L`VQy;3AmeXwR)AaE?^2QK4mI_9(yJf_fWIt{|@`88Nria-+Tvo1W6rn1E`m)'
            '0A)rYjpO+!09-M*u}m2EoKMPtqeln!+9A^9JWWTcCfcrD8e><TPdIXGLLw_`YLA+5r+ttE<_kd~0fFHhFbNV$#fT|6<5X>*N`{np'
            'sq)oF$F4XUFOvj~co?HH_9Ch9guX`!^=`dz+Q!diek3Ks9UcFoLsX2w5d58oi*y!VLJ0zAFsmm2k;<}c{s)YIjA!(xl%8QdUG9j2'
            '3sCs!H2A=A9Q+&BG;MUC-9t>~o<%TeI*$$Ay~FS%*F%Z~nJniZ)FU+Ei%|;Nkn5%6AW(Jgu{KVWb)S)9@Ea!oDuszKu+sb>%oi=e'
            '>s0O+?rYeEqj)?kdcXs~;}q+@#OM?w=!W4g@Wd$>Hu=Y?i)8~_oWRRNfiv{m)Jk4*N!j&H90UJ}DC@6E!_C{V4N5Z17P`^WlQ!n@'
            '(s4#hw8oNRhBd-j-U%jr>x`CN5E`Y(NR5tyXqgL*0K^1+G4S31pc43oj;k^i3)h8}X08h|L^x%*wAAFgdt~-l#9`QlU*ETQfowJJ'
            'zJy0+?iuLcF;R&f007LmH5~+s;ZIGi<!4&5FMLM+G(ID`LN}8w@|X5-?sOu_x4Aqj1}4{MOjlhg%JC(UK;KAXX640w;)$TJ5ehKF'
            'XSO6<X*4z#0w{}>U<r=;`#{FvxkNwUS@Z|up!)sG8AfI}^=8d*x>aFp#skpD`KM&G`UDd<2Cy0cY#Xpb{R?<90A)@r_Rpo=-LEC;'
            '`6$Z>_G#9XzKv!0R=8i=zU76bo}Sk%(b6i`Eb-4>jXl+ae8a&i@<x(^WK^9r%;k@-NZZ5LuLKbne^N6Nte!5YMU#@`A&tPa{Qw<`'
            'b)mFa%z309y6MK#rsAs7(DxTjeFPor@O&_6!?yrH7g?xd`BZ3>t=0JeF~?z?`c2i;*{Z>LZ5I7B6tF^&^EA^({lho?v#KQ1d{ao&'
            'RVJ54kJJO(^;KEfQ`1HIU~qPDF&Mf(!2x^+gAd=H_0Qj)9-#~I^U~XcQ+RXw%L!V)puyhv;#=`@SM!~};53ei4`|Nw9<$6~?EoNn'
            '{qy1AxQBHmkt^_hFk5+Kf{olno7dJjI>ii?s;jD2>!BB2Fc;nkS%b8#qQAnOzSdJ|c3JVJ>t)XChU13b2!og=;u`a0U}1>~2Nzs-'
            '#2{nZm$J13FE~N?p}5(I?#|WicvbK$oCqx0anG0Ssm_y41w8m36K%$ZYp}`+#~zSqrkQ(WHKzQO^nK#5B&e#{>TxO8Fo6m!BY;qO'
            'PRv8{U69IK3ZH}FVgg<+<16D;i?1t`U!h@>W5ItTPAjeWAosh|s4lRtE;72iE*^CaUuK?6o^tRoPc&~IM;&DkGec2Rnq&8$)K>Hm'
            '6`{=QgP_Zg3L}!OuPY}=uDZ7EDPJm88`b2aA^=&BiH=Rm9}o?tc+zC;4*zrQrumlhBuVtJgDmTF3~!ulid`+#VXO@Gp;Qid>?tEv'
            'Ikc(qYyYf&B!<1W$Agm&S~-_v`2R?({A`9XjFbxmt2NG-T!?X!r?n}^6Qk#zz|=PxCc?o&=%0}K#z=o-I}@?e?kZG6|Ah*ZBq-H|'
            '6Y4aIx$-_GDN!MJmBL6)VpfIRg|1v+QzrT<pX@vX)5Bx&jTbFpSXy;4*z_?cCxo~fvl<2~X{@=@WIsMT&OSdV8wcC-FlMklKPkrA'
            'yPGE|sQ=Htw&D*-kAj?hl_T(L`yg>p@Mo7Ok89FY%V5Q|cIz;}sS)QT+2V2Pn{;5mrNq%JAOWT>bT$moucee`)V{nVB6}v9OsQ~>'
            '5*5hEuvH~LYUIOuSUK|95}&FVt+1-f{$Hl+m*$DN(sM^9V_WsKT7@dNR@@B@$^Ks1K%-l>@~;;dKwE`aa8}#TX8=3`RoG#!UoNg='
            '^Mt3%%TVyLu{$K7bxlC(0NHYl8=ZtoiFH*MG;+j>NXzxyM(1iIuTg23a@lPXI_y@o=9lh0*)Z6AUr8?QuY(|ERCn6wL#1g=$P=!r'
            'sk{)VpGRIO+a+(znsGRd^4cb#1Z32pj!opPfVHk|G>+pCU>LdNn_AzFS*muvf2z3=`{z=jFPUM0q5$ibB|`{ZopX`2ze%HRZ8ETu'
            ')G=2T(pa+u$*uFWeiZ@9)ig0b99G(6#`(gbswiaNHR?7MUOMY~XX|=rPxYq!#r5BwRP?qp8C!j=SDZ!&@2^~AOv#0c<~kP@B6(KR'
            'CV#5@>HL%?Ajbx%0O}Q>CzZ7^yDibIT`_1K=+2}^t~|q6TjfZ49aT-cnoHx>{FxrhRg}3H>pun)$}NX<hg-Zpgf<l<R%VBj6vYw$'
            'f)^jFDuR2dTV2uZ$f~kUH3*=?Zq1NYhyAK#Xp#5H=A5f2v#o65qfIQD*lu0oIcnfk8Qo;p+AFW>&!}EcUVIe=MotnZyE`Sngg<x1'
            '%iTIRUxs%T(5_gGu9RUm+G5Gq%DY^D^49Uk@;K||$uF_K&s*E?SywqN_NT7F5C{{$Jq`If@b_8-$Rtc8k<*w~NQEs?y<p|!xm%gV'
            'RGOQ9!FU%ogdJa;52<{^$VY!})B?~tMeCq!VzQE@G=AZKr;#ommr^7PWhmF~{{@R0sBb^_&vhX#S(qiKDkvtQ;Tj31ra_oLg(a08'
            'm1zKPIa(^bMpp18>z`rKR(F+Unb#1p%d&1g6~|jR8Qx8%P4bs-)=*3x$ixPbYzg}@37#w|)GSJ6<QadLVBV@U7zYA)$gEM!;Ksp4'
            '7B!}cmNZ;y_z5>Pd}Civh-o~h?vjUlZ$yi&kI8<YM2)o?c`yDX2dMv&&XD^+8~G@jqaX|5xIEB;RBGH$qj;sepdw2z$e`6;-kJ{='
            'S{u2I7Lx+wEy{Kz23gZskgeh=nE==ow5)cgZ*ZFH%)5nCw=X1abGX$c4wsZ*m`2~O8h!ifapwUBe?&(Y1H2#yuUpK#axwGui_z4b'
            ')-&H|Jri#h%w+gj@Z^H{3vx|!T+^|!rfl7svdT4Oa!u7_oL>F*yP4g0b;JJ|zyB#<;v(%9u>SKQLr4rQNLY~y+7J|Zi}W_mHZ*U1'
            '0_m={j+e4KeUdr6-Z=3j{?&cx30uS`ZtGL?Wvj*dlV3YiTOZqhT4Q^vs!yGs{8U|^It}@$%06|PVzFPr{Ghv8&&yvc7*O)BSud_i'
            'cNEH8R8r(hc6$~l<23u7@?_DzNS2doS@K(u^JbjettIkQWfjSlYlN96gwd9;%{J@OlNMrb<?W}uZFjR<OnM8j?$+M1*Nm%1&^v{L'
            'ue*x>eUzzQsC=4>eb*IxTc1ty*@Cj$BKP?k`cGDq4W`6WLgjocv*bGc?w`o!gYvM2h}GKP8haxzR24F`UpDPOoh>WmVOIUElAX8!'
            '?t-~*ot?F;AZvNFMAz_!s)3H5E-F@ydTqYcPaA4$rfAc_q7|ve@@5WUnZygwfCc5`iL|lJlesx-Hd`9b*P*ffX#ii<H|$S*BM!MX'
            'B6&}dHYt<fV^RfSq*IalT)mUX=0>{;S9UZzZm=fIM_ZMsZ~ya<v4qAZHa^uE1o;xl|0E<Ow65<^MpLrvnJ(qhsMoDx98NVnDSEOB'
            'OLWGg8mn$m_0y>)#b_BWxu^-IpL*fEf+uK6!FVwE`ot3|-|pP|*m%mijqX*hf|fr1wQ{+wy?>HiUb`_l(b?LRc<AYy7v5Eqma<Aq'
            '8l|Lk!>%EtmGX#cG%8p-C9lKLW&RhC!6<t<J8efgT2D@vPA7iDFpO#My2$Xn(@Fl~NIP}2^PS>{<Qk_u-CCB10UOKY3RG5+G?>V0'
            '_w4uvbHdv8@?L%fv?S#kr(Q+Dh<1AKEyz#YqeOt!#xs>snoWDMn2lU7#(XcDr8Ese>KpOw6<$QRcYx1i8^b((NeEai0Hl$)0gZDu'
            '_Ql6IK_kOtnFyVqd)pT?Z#BV$=t1HXU4inGhY3x}WeAI!ASGcc2Kp9@9I7RjS}__?JN~Jxqm?b`BDTgsp`t#fl^vd4u2=^T6u6RC'
            'A|>Z50%D49CR8F92!a2xV-+gYmI)6XCsRs0uJ1C<vP|R~!$nHV_n)>Ul*i(%>NM*9Q}?HxMZ9Qo<6@$ajXhdaPG{QTlIBlORrv&_'
            'NYZ<k)gAU>**BeNH`RWgsG;x$RUR}WvO}RRv_r1T8`YMlbU~DBh0XW6_yXl%P?NOczB_sHCrQ!eb4yb!rCmobb>o=@qx@&Swra?9'
            's~?x(WdCVa-av0u2`2q5IdO6|DrC!M&d1?>YbACKXX~y6V06v|H}N^Mp)`U~pB52Vu?Rh?dkcM=d6oL}!*?>jgwrunh;p1x$tYe-'
            '_F;Rn$*&bs<M7QqY#hDbGE!wCd#0CYhpZ^$&P`<6gqgpxb6M@{3@P%vW(#r-i{ig~`=bAc=X)|}*$*kR3f0H~^y6)HiT&YK#u$sn'
            ';ax&ne8X3#*2Ar0()L>nhRV|Npj0<U=G?R1s$grm;pC%4lcc4|OOpIz2dNpa;nBpN2cQRcoEoYHKZ3{@Q57D>jk+`A86FiokXWY>'
            '-okgv7E#J~aF8s4ssHDT_W0IIQVQ>4WK(0bn5&aWqGQA8rfvBCSlleVWMZNk=WeONv|Y0tlYwg3shF@w=DIM#?6_4q%&t(_=0kI@'
            '1*bET>)qKAFPyKC0ppB{YH$T#PFwGPAjLxGtWU*r@vmPaaXneK>BEZUQ0M{G@W?p}5>G97Vv<3?Ukn#WK#EIsf*!LBPc4sNS||H('
            '`aG_4;i?3G7W=3l2s4s!N69lZ+sbQs{Z&=pkwtEY!~XI6;rVCt!;P33>Az;e3m~)0Nsy6>ENcg0Mm#zK$n%T}LioD7jS$}E3BmU$'
            '9)v_B3{%0~!Mp-tM#&99eA7i>zEh0E7)nX}+cXrNylV*)D%-+<IsF+lyuo4roGW2pzfw8xRKCYbKNPH;Mu|U`BUom(<T~WBrD%+A'
            '<6unrNmi-2h1I4iQ1lnnNv<m~pesuw=|~u)0vU+alz<q+U@-AG_2Cb4mJ13l71EAIA;KIjg9-JqMh<|Qvr5w?eknONRojsTK8=z#'
            'X7;#(elW${kU=!}S3u<C&^H+N7VL`$9PVEdyrcDJAyz$tXR=DMF}m~UG#s<csG31jowJ65rJcyGG?g2FjB>?CA-9*tr~QQw<H&`{'
            'NG$t@$Z-Ux!ZF0^Y-4by{!PV$toP>Yyvk680H9hGDYG%qrTTrfWx)e&d>RBM8<17mQ4m1MUk`;LJdRUV17-Wl3&V|Stw<HDRaF?C'
            '%01EJLL5WxORG!2Q`@|va}>L3eXd$Wl>_Pe3eQ@{X_WTgqyng}t~>wRgtw2|R~11o5-d?L4_RpoUkP10mkApd1e*t3NqX*m?34#_'
            's=QZi>2B@T8fS-zl|<QMwc)O8+5gMM;H-b>_New6cd<&~$x>OLP*2NzYn0WZQMsHJT?Sag*OuJU?W~ICWS`1vCS=z>r>|dXFff0R'
            '$cD%23KdYDr%bENK~!}lKA}Q1kzD>vUXZJzB$39~VlC-HC9P60bA1Wvy2{g)Wu-rPo=+E`-eBe%*QWk-FjiNw-egiWxQ6z(;;yf0'
            'TK4E@N~~82Q%yXxDrWz#b*m|XI8dn0Mzm-X0LWmVH7UEuR;W!0MIXj|Ue$u4Q@$!76K>g6_)}KSKOUW}C*%L=^p~r}nW|E?$X%)}'
            '$ffD__pY`oo+`}<?`F$lxZ_-HZB_DCi>Gc`_D#iGw=9nk<Jb`#tS4sM5<k|zi>>e-(a5a|&MbeK?=+j172FvVoOuyAw|P7orG7cH'
            'ZWKG)+`Dlbj7Pf8U=@0cCZ6*hNt$$O8!-b?N<pa3s{+ebV~)L>M6+Zpyt!52#g9_eXS=Q}(ug@6np!1QmE}umS-+*^wk}rm^y~uL'
            'l{v(8#Zq!L&b8c;7e&(UH1|>reb+JcTmw}q4F`bnXc&}$8XPi7TErmUtSAej&lykKZJiB(Gqx5I#!G~5P(&$bz|@O1c(vxPWOHxU'
            'RfUwkV@9APPX%NOkc`JkFtWnH>1DH6v-IhJ&@~xV86j2eGrJ^wZvSo!Nv(g-OTFC2v$%cr_?vN)HEobV0@@J_3h(j{LxbX?3T=CQ'
            'gQUwbAsX=|Vc~n;y&x14%LN9H!pPf9s3zMoGd80buX9_CvMm$O>M{#8B^`^X*vEW9m^ha#LzG1l#9~?T;eZWtXQ~<(6@M}-F6H)6'
            'X;^Z}_@+Ncm193-xn75Vl?+(;kg?_#wv8?~axqOC94Vn}Y53e8sSwmo1+Qul$)55b-HS>faitH3N){wp#Xw}y4}7xF$`P=oy*dL?'
            'scl7xZd#sVO4P_g&xSmH+^)QnY{%DIX%+t{L#E*AqK89+%s6xH0)-_6Dca7@i~s@ZQ3@20_uDda4V!9+^JpAV@4AKEFeWXH&Qsr?'
            'AUCE>p>EpeT49nLtei%@I7Y|p-+hq%<Z=0(7{-IT68SM{Qw4%e(fD<%7eMs0;dGXO21&CJKb8-N#N4jpVZOC0c|4UQui4<zGiXA2'
            'e|GwI@Y7(}Kg7R&IyyagcLcfvmqR!h411^>mckP>Y^}9)AB-tIHfmCvyRsU*QFC`1+?Zz7^8KAgc}|S6LHYA^1-x?ZB7?vJY(Qa+'
            'Z0S}#-VFxhqIw|h+I+ml+10vvvPjeZ`4cXaE0kaJ8n5{#3HR!{rt<K$u9qGHXT&$avAL8VUYSz5KzxklH+Mu9FWiOvcLx7zEaS_P'
            'j(9KAqs7x{Fa}lxl$>sypNpF%G{p2K=7{NSybLE~l0@5JkfrKwNbF$}IOV&5Yog4|0-bk!;Z1ZY5P`YXQO1bpL6%p(2~wHdkR-Ev'
            '7(a{A=ou$+K=_=Vl1B;mBmtRvbC4)7w+GA}Jqn_6xSU{Yo+Y+C9W!s*B6vSobi*5@$Gj9n0XuTM=RTJe1)b56MG5@Yo4n*e2^%>('
            'jSk4ARm$Md#tkmiFz9_Fc<HiOqTIvY`++KSl={GakYMq5FlmwhirSiAGz`|?01}4i<U;xXDLEMtNi>?PP%a`RFq~&Nkti6H*TVtE'
            'rN^8t6vRzDG>9OiGtvWcEn*v@?O;UOC^RU8UJt+z=$j7>dsxsY=wH311=?Y%xU>mu{&E3dE)IqlXZ=H@t@+L<y9t$TFHcF2J`d&~'
            'Tdfw-4a&os2V84;hm(ud_T+grI0oH%rk0S?ACo)CiN85R$1|9=v?n|{^%^VrgW82hWw1Y|Iw$$pff&I4&O*SyRh)DTIY)00P)9xw'
            'mk%n<8dpo9BsY&&YDGjii_q+F_KzmLv?L*ZuNVlqTe_586d5`Yx2DvvT}%b~p=COmx7a(=oklItrBc63lx*xRx<BBLB)Tb7c`}PR'
            '-J?QPtbC-)K0&uvd4_tt*?}h{^0Roub7L2?|G@OCFd53=&$m<8Fo|pPR@azywIPNI&7cJpghE-HCGP^F5oP17G8@C*b?JYB1SWv='
            'GW~sVOqdV@Nqj4s&&0Ri3Jb1|gU@$6-;3{Xunv!e+1kZ3eRO%eKMNyDT5Uj+-W6)tz!P<!PCBn&d)rFR=(vp_5R60gD3vUo|8S=s'
            'rOZ_i3;K9HgV!pbk`TV!mF}0j)BQS^x=(vK>0E8~x#`Rq=FYx|nZUNyP|xynD50TF-kK51F2ZuOPYn*K3lOOr&x{jUGo*A^a+Md_'
            'r#g2f%P7_N+$c*?ZeSF<*a}0G11ZmR8Ojo#(x4;7#A|sbntTLWhLIH608Iid1d6X*<CM{u#o`zon59ug83%hMu8l2knw<?kjpI@Y'
            'y~#Cf`-<5K&#1EC@imtuE0p0rMWGf>lKNYgy+tkvA}327KVhwBEaDSH=x~Mi=v*c~NI6|3USy{-Q{@P-4@-apR+H6TB|(<CCw(ji'
            'IWz2KXz5U7sx5l*TnTB>2Z>;CFg*HIocI57!P&~-4-^^B`XDHroGXEhN*M?l>5&UdJrR0b#UABYm@mrs9<G56aZKo0`10?9c*(c`'
            'D#0>{Q10hQiI(9N#_8tCRf#h<vL$coMO4H{NzN?7WnLKHboBDD@+9Ww!@VQPyHK`JaVF+e$fB5kwYc&mC%MlHO!AP2dPopMvCaXX'
            '@>g=vq>@f5A|X_szRI(uvuoCVSXPzXVXsHK1Sn33Y!>lC1aq>Dp%z7M*iv35Q(#RR4OJYpCBdD@qVuviCvHFdp>z|b;rYZ~?C$Nn'
            '*nO_OQ*tgu#a^UwPP=&B15xvl47{TGrshB|w3|yEEL-}TLzSr^Sn10|7E$sszyI(-kEwQ>Rs{nUvfeUoo(Ln5LulG=Oj8}%RG)fP'
            'Obfn26~geGpY*kx{GBfWDbLP%@3?P!qRMSA!&X+fSN10O!1?g>{dq?wS|&@XFHi4|F6Ro68LcpiU-r+(BbvcVs}8i0+3+$ri`#ts'
            'wp(&V#H`fErrs!_qThv*VelP4$=oP*s}oI1`;KCTs57abh!Q##^HwfA=sUOSYdW_38vf7f6FL`;k+ym(uYspL>PlXO`KxZF|C^od'
            'N9%80b^yr*CH>2gmeh}y5{&pCo5fZ;K2<(#Rft91@}t7|pX9%6gc(Y~6`7M2icoWTr%({@xEO+s!qr|m=UT;mopRnU-M|>F)Ll$z'
            'AS~n?tskbobAJ0*)mgo*=&n|-H=yA8)8L^f&m%P`!$Iw^i+-{2nhkCh?`!hEHTck00N9Xd%5R@YKW7BF^0^-Qn9Tj@c3fk&)C>iL'
            'XuLvE6w~N+b}R2!nGmhJLoIHN^;%k(&Dm+MerrmPTh<NW(->pkp3@uC5{}2@Qmuk)BA`q!$q>mgya3;t=4PDS(@QJd7Z+|Kr7eSw'
            'TYxu`uf)^0zomJZ8+FCk{n4aZsCmE?w<VreWT8u`QfnzS54?!OJM?uY0UobH<Rke+OZK*sYwEXFku+XzeiM1j>GCn00b2a-{V^~H'
            'yKwZQk<IsbaDF~Gd9%sBZc^d&t8N5c2gYeY9ut=P7f9f9GQKwntDPrOv*vDAx}9>5`4SE@L7p+JcniZYDU;hP?LJd>maJk+qRPuA'
            'o|WltH3?}@s6x1M<z*WA=3)bQr>0IfXJMGSaMmgkwAQ&Om7`PbSf=Xu)Iy>rJ51Ff$JvUwcsX)S5N)pfif>3!13-@SrRJ9XZFyR%'
            'xy4mo(%mO|>9y;GI|WB3CNBjGjFmMY7sIPx>m9NlD=Q#dAPOP>)=L=|ZN&9{a`hWy4y6!tHeF-p4Q2Q!5YUnZq#{I|)rW5F&3CbA'
            'xZYHHUz7Zi?CIw;(it~oySnUE9+Ax_W;Fyhx0qHIj6llYvZP;h<S5RznAWUB`PF}Nr>F=?FkO}CaCypeKcx`2PbS{>8MY%hx9ZKN'
            'Eju((-MQw`iFRD#=gauYkk_>hdgY;wb7o+MoeO~P*6VVFeg?(<vXr};f)gZjqAhp2NVD@ls>gkF)~=doO8H>qT-P=?MH4iT`S-5!'
            'b8M{bSP)Y1#|9rVg_CAI*xp?H+&?)Qyb~bw91ixibuaU80vK@uL?sJ4;?45TkHTO^@ksC?ys`Dtxi{mG_(tsqrVOT?R8`0jha_eU'
            '&IU)v{lfuTASM(tBRA=>daU(ZQLUe>XEE!>`&_ZCGH*;SqE#KVJS$A^V)(XID}y@!dj!{?DXCWeKRimTn$0k3W+o=_O|qn<5w{+u'
            'Kfrt_WK4DTOF2<w&-zo;(8njmfE-M>ijfrp^geHf6>K_dkD%$jdFzJ@=C#|Bae>CT?L3O!3jJ?`-+2->9_sotS9y)H#{bkqthwb+'
            'gQFujtnObfPKUk!87H>B_a~iXnUJt3uVc^B!!;w6^Z_(f7N7lBv-5uz(vz2_LPP0}yjBu-{>tOvp**5!jwERf!I2VxkCmuG@{a*z'
            'g0tYM?pO+o>MZ2xUDgaYS$_1xELdP-?8M7J?OOQ}R(%C(7Ws22P=F|51RAaxF7s!Jm*5mwUb*p!GUmpwwdC-BDczS4SI+xqKljg+'
            'z%m>h_fIc|T#2fzwEwBbouYCo<?fZ5GyYfQJXIzKtev`rT%A+{0X4(T%W&tuSLp5m|6dKuvbm?0Vc4kv{}+0n2ED<_&%L9;p$nMz'
            '{$fbbt1t=uN$l;{e5us77FPOhT70w7O)wQcr{h~*hh<}GRhyi&z+A~G=wf@TkiJp6Y}lFirHP|9r%);W!n-dN#@oouQEB3kHKaf9'
            'g3?++A>q<yCJyJn+`L><K&*U0m5W*O&y8iN&zGS-M|OIwhT#*kl4>R|Ew8Ay5ZVVTTUl3_#J;@y%7zumZS90omi$z_lV$cwCe}JU'
            'Kt8!wIhuWcOAC@$E$ACs3Xn41l_`&QkLrp`PkNFC(K`IR+;8t)shdgf5^?S7YO78fDBdNLSn^5*Ki1@uc<?T_X)m|09zihj=%!&i'
            'M_kYl8~I$-63P}ux}1%pu|EqKH_-83-(jL`*4TPVoQCCXZ6*>o8e3mOzWNe!RgFCL$yDg~YkW5ODKi^`7}<<zYa?y%Gbz%iQlrQ6'
            '<>~L)<{GuE3?0)=re5pYL~ENivgO>mz?ZdyQ!fcHpHK6_pikQ+4B5VN9;Lcug@8B4!7K~J#jl4KXBS7uy~9DLVdNqFJ;+=@;xC>L'
            'E%&Z%OFuh#K|~?M<X-!M{ZXYG-Fg4GcQPCtoPU-4Xg)^^b7@shiAXNP6xM4bG{TbhK+;mfUH0G3PfuhP#T_q6ycIdH##4$YRq@5r'
            '6E}_LRI4Zz_ld{8aY4ppHBKq1h9|DCuW=S)ibcDfjyS^`xl~lTBda2}vdR^E?unZvy6+nC806L>SEIA!%D>Bba^)3Z)%Ti>QJiv7'
            'Y#B5oBSed%v0nm}9mimx4#&<klLoA=`Qtt=(E0JV^xJ0VyB{5R2850C^+DTn?T$kOlydyh(+&pnESh_c^+i?t4`V=@0dcK>6jz$t'
            '6b^N>@qn?n#^09>hb|}s3|VXMIeDT<_(QuH^1#?&LqHLbGLO+?6vj78^x2V6^I6dMKJdg6p2?%0nS5}CQ~OQ|CKzl4-Tu@q7Assk'
            '4de4OZ{bCAZyuy-?`#JiI`Bp%ZJfy^e+xc*WqIe+-^#9=&6KC(trD_;OR)fi4j2Iz&r$kDgsr|aR^Ih#lcmX4o+yNqVs@JVy78?)'
            'p5af&$c)Mz4eBC#uZI|gN`YxRlG5a_rC0sA<>1f0!8CdFc)(2groTVE-WS&o4NL?<UVx3p<28%Esmwnrqx7IZtUU}dPiBbcR9y4C'
            'uPG%r4E*;cott<{9h;GdKHMO8O~_AILcn3)=QwoWD?o=zsqP9nGZowl`B~TRmdYf@E06pX{x4_=EK!g5b{Hh6rqgj<oT%$QO60OK'
            'mlk%*ivHt9GUks*s|xX}o1@IdEzmqE{8<`c@(UXCUy?^7vhNN<1hqPHNiD9-d2j<GzmZvUonCTgJLx%}aR)_yoX?&!P}K5_5qgD<'
            'NANJF7!<MScM{KFiHH`>62jD%T(9)3Sib@<g0qX>Co_ybi9@mdfYW{4raLbIJgNFz7t&{yij92QN3{S2pQ6v(Sy80sDT6JEwaar('
            '{wz-u5R(SgMG%!JXa&)#k}k^bFZzkInPsi80yTXL1Zpy0L4xZgS%6NQWFN!j$vn1VZ}%(Zu)X!3egBfdrIQoulrK*u6NahXhQX-A'
            'qclgDsp8iEFbQrznP@s|Qw=vMY;&~A{M7lZoWmG{lVn&&D<#udRsPI1S&d010Hy#q*K91aY5NBo(F<!M;8=^Z05}EZrHMgTHUPOX'
            'e627Ve~BXa$bArdV(PLH9Bd#Qw?Ga<-_A`#d0voS1)!ESIfwVYY}o+~Y0`Z8{rAsb!imFeut@+<Lq$;2NOqttzb~&&X5)}I+Y&jO'
            'D!>5M7?t7b+m^7RaTnxaopIfbbw`iQl`_LcRnajSVWXNnnUO;_XGm8L$ZF;01<r&NgL7T9a#)>h$xW9dOmG2A*G>tt77UcR!o6Jr'
            'fpfhD>Z_P4%8dV&f$?U#din8n0i|AvrXV_TOeCw%ZIx&wkFMk>NKn1dim!uAl@h%}7{CE`JBZplXhLkubYjvO7g;(dC*cT7$|jHH'
            'QI^?loCF_PaJ&V~MN7A5_y)o}F3yfxth%8`r?0sx6M7cnC{8lZjK=<_C^|$pDJKo_LOd8NagW~fGJ=)^Q7e5EuN}E?D&{g>MjYwH'
            'g;<1-wNJNsXE47Od<?A$6V7Gf(wDYhQLc=bmqr#S7FB7kDu(x(Jfhu2u<$WmTu1G;{9&9l=em%8O*yp6Fto0UM?=x7<Oioy=gG5b'
            '21zJK=UL-vwn(m}YX(9ir8M6-Mq&+Xrkt8t0<Xczn=H_+P19`Kx+<xXu{As8eG$MsGJo!Wd<9#932Z=JD&_qHL~du^2t=YlfDSdE'
            'c%oUmHx3G97(`ME4bd~Q!oUo}V;PkCB?GfQAgtV#-<Sbbpja;AAj-(FsOYfqK7Ux(qB>-)Ci<*ArOypOs~br*+cXxfZk$gHN~;;G'
            '+_Yl6iWs(v15|9}tch_W1*ANN-OO>;M!~J=u41ODV$2F>d0nWP`8VYgsi7M6;b*`GJcdbQKd(cMMU<?H;ME0{3Ndr#ubb~%7*pr0'
            'i|;G<FXqYHB>JyBN=;K|gBZc52jiCJSeA$-1tZ)nL-x0tu<6cxQjRIE45%!B!xF}*6sG;4IIK<S%^BKXaE!6m&e17EcNPRReXz=}'
            'tfQ3ppi>6uo&?+O!e@dpZtd%{e%~Dq`e*K8Z`i|1e}jX*(XTqr8%<f-&1ub^tIA;HY&P+4{DiJJxIWmh_x5;jayU3^sls3@Kbrw1'
            'Pi2z%39=>?yQx&8YAG`h*O9F2vSGL2WLh@6)XZF6ljV=NP@24p*AKmNz%31g%S<cUmypTs`&g4MO0A(jVU%j|6%_dQe$viaDvFMA'
            'ga%CzzUKhYNvak47AkUcsn|o$z9EnsTv<>BH_ZBS`pXHKuO_nI9r5@M$g2JdvpV2mNqm&mVK8!+AySdpsVbDO4TY$M-Wan?;4AH&'
            '#7Q$yc2}$chNBcK^F`=$ZPKq|iJ(*NqR1ihzH!ny(kyH9HExvpajjLVHs@oW*{oZwynXvE1K6G2d&x~I#aCHlxm!~JiWh|!5Y_L?'
            'D|KqK50=d`<tE+oGo4z`$!lw($-OR3kd=rH4<wGt>o84^n`bfXR%9_O3_oo=l*fDgE&45b;4eQGQ|4jFUUlx!nv@j0!Xg3{{<b&0'
            '0u9TLst~I5#2z=tH2tep8E($$?r57jH9120R<xV1P(ih;ezYXSy8MsC0%g?1JN75cy0%fY;H=YJmr)S~LVGMhmaXd1SRjzq>a5_$'
            '{i;AmX^giHrBz?D+C>g;NsX9lRa&wx-5h6{vs0g`->ni&P{k>iks9eX&gSu)eKJs6mlQ!|v;9_f+$2HRG>Cw)z<_NYQSgYO$#v0s'
            'twK)CE($oyxiWH``C~u0^HU+?vw;vK@VE4sr8_rFaX5+ZQ?Vn`F(4X{Esq@H(=o`ask$jgo}+?qQ8kK1GxcEtM@Of>xM#gzTntU?'
            'xx-Ue%S(j>trfdfKorbV+2D1;Z@JeM-58m$^;VwISoXuJBB*Dr;I%}n#lCf+r&MbG3@``O0r_?cok<p#`rVa-?r2g*bS&I4I9a2('
            '-3^1UeOrqp41Gn(Ff-8a#S*!n`G^`z@~rF|Zi2jy{nR~>YnlfUU}JQX-CCP~(8Y|{atG8mF!#0gMsTn&_=~4uUV`3cKp!o`nJWXH'
            'WpdWv0ruPA=C*wohs(Kd=D)%ma|-l7l@DkT0fh>cD3PFVLK5E#dsfgb$U=XIOC*q8l#H;McVldD%Q(lJz%TT@Xt@9?Dz#Vb#Vp$!'
            ')1<p;;174`4;)9Ih$8h{Qvu`6Hs>x#JfQr*!IDMdXTNuM;K9M!U`P+`LD9*G9M192l$;R+eca6Nq&Co@ym881h~(+C)EiT#l9@L*'
            ';+Ei2wY<#$!x95kkC*;HHdI&~$<xxSq?X@*BE+h){_=eRKan6gP~7Drm7p*a&dKD|d2gz_BL*m#Ol0OePn;hE=Md3yJ_4}fcEk&r'
            'fl@6(*DAsC=iZ71nq-c;3?&}-uum|iw2I3l;c|;XB(vb%OYv*8%*a!WoliWY9wSZU*LXQxj(j{WNiZf!MFL@n2Q#JAX_)pmP<h_Y'
            'xiIg=*-^SP4tN(+Ql8{xwa>RAGxQXY)6v@MQ~XeihJY9Ob*D}^W@1wbCn{0N^k&q9yIen*eKOS*d+BY{vaOm9=T6<Ye_s-yYQ~js'
            'Hy+o<X?C5r9$Bpw7~uV#?_~u?(#9Kwj$VQR#^=4m{y|@CkFeBwv;?%kTdP}dMCO+XAVjc@SI<i{yXdUfiRD9T;m(Y#HWW5!BhW|q'
            'Ttpaz0eg{H!7nVf**`e$|E=J4YI}~F?v)k0WBNVd;SP1#Z0&2A8n{aABvme!*Etj~uR8hYifyPW1h3i0$V1P|X^f|w5c)=6DSCmp'
            'exte{tKMRVZ{YNs1H6v@#UEV@^nFT2^E&?~7dL@v-20=>G6{Ah=ZoiF5M4_lQdJwdyvnn2YcJ6;-C<H%&OBAQT1v^m%Y<C5*MqD#'
            'BcTm^)Au7;F%XLnI_qnNSROBH>xF1OtxKSog+j_3(<l}1RB!gzS9r7B3*@JC4R-TOydBjgZ(xUviDE<O3J(hfZ(tMiX7Czo&m=*)'
            'Dh^H`+rpRdA52{jVBUYamK$fO^rMb9nKYrRJ+j=iH6cnkn|=IdRmcxwAH21u&krlyiyDlk#rE>4QyCg0QA|#=_M~fix%Q-E<*;k`'
            'sLZmjdR-e8csR*3a>q1XqAz0=Z<6SaUeJnEf(#eTsaEiaa=(scN?^>WN|~4MJ)kl4SwAmME+@cOYnLFPTy0RkY4ZbvL*OS?>uWew'
            'ogBv10-&l99gvehBHb(1qDuQOUnpFoPS!5V2FeUAhQo6uM;$u*C<*o7_b{YYYTeodD+P9(0PYicG?8OoBBVw2P5*dsGH_u;_vh2W'
            'LH}H{yvE=B#qBDg@;=M)4F5g2^%9ad8}INM3jTfVM@c|!j{J#-|DFQB!+(3RugmN<4ght6B)N^LQ*W8Y5&!IiHXV8s?lSNee#Sp9'
            '<|9~fNUg)0z#n;Ps6*Id&{U~2a6!<SWg|ZdMpSUfOFj4tG$xvs2Q4OkKplF3WJ%;DYOG#J(NXy79P!VimAvuB$?^lQJH#Fd=vuOS'
            'kdRNp-+*UFK7MmK3e;#nV|jyl-~lJ6-sgZa<J%E`8vh7E@$%^t=gZKWf`HN5a<)|BeZ2)3<U;VZULEyL-dyzF(Az9&56>D~-|UOS'
            'c)Vm=yONm^H`s)@FP^;+v$r2xV((>p6p+CdGf;c?;{Cy~Xu|I?m;3V*;E?!{^WO1$ENzUa)P?Tu?(VMn;+MhU@GZWfhX-$aCnx<Q'
            'M47$9WN;Q_NBp2{L;Qh*)01KUudt%icl{F%z=qz>pV5b;eP2;A!ey5n|Lj6y*%HwEl3uh7|0;R>{+Y_V1raKArTT9C*e&p4(+QFq'
            'Y{<lH*3Vs$eUwMX>M|8A^P)4Q$&`hHvhO5wf7^GmVt%OJzI@Rcy?in8$(a(LS7mo2lE>RlO<FYXttX=`5a%fH1MtmLpl5=3N20Tg'
            '0bw*-jOT6^yZ7FmL@{`<FBDRFNI_k+mv;;20*%_}WfG=c^9<Cu?8$-dgGqMVEgmEeBfu|ePW(&aEmD0%0R=sAHhBGd=a=5k{rt2_'
            '!P+?SG6qR{$zf;%4$xv|eBFUnUSke1S-f)u8-UXIBJ@^xxy;^3vXB79NcaoTw)2ZT_CRI34-$?H#05~1t{waUsK#imLS<We@X?w;'
            'ZJti4`VCeXh^a&v&A}F?u`L0b5&~+li*GfY;in14yy&0ng~1I<)azO@L=gWP2+4-{POhV~05PQ5_|pIg^7EJQm<+e$IYs@98VK>}'
            'JQLU<ugF~J`u6Pr{=b0~wuLvfY9bZ}wemGqV~MetmbCy_&kv6E{ShDQ%0=k;Ti(gM{O=Ioi-A2%V6{gw8-ap|&qH@__t^{dBweuU'
            '(V8PsVHZ-Ex`QWuypskWK^c>^6?URHYNL4_8UUV>5m;Ju*c}h>4WKWP{5^&7kobc8rxp}ssUJ>T68|FKpHO*)>6G)SqV<H4Qi`Z_'
            'cOir3sLV!?6%dgsVRYep*)2P0%i^JEqNWOA{s$07(f<J7c7l{4ha5m(7vvre`m7>_sY?mfAn3SAiqgH8q%o-e2Dn`#!WdqlGirU7'
            'sfL2G%hB+6deBFCi3`&Tc|o`i^2JJc!D@8ce@d=v49AXAmG|c6WR27c@+{;}Kn6+O1?DlK_VUG}>HW{(zErTjD(TcX>om2TN#rfU'
            'zU+*X=0&1k82_r!x$?@<h+dO8U14gbI9#R_Eia+Ptcx}r{X-?Ia7<8Mnm|+84szb6R-|#nY9jtDUc9i{sr@Z9>*gmjFK=hOeer^D'
            '9{;}Fe{qF(=ZfIz=RfT4J(F*cx*50X>f<)nUP*uQ++&6b26@xf5<8dv5BHjjmm-x~Mv_ixf=Ls4w!8oQ<y9^Nkk1qXprYqMb$YS8'
            '|Ki1!bGdtER3D?AP>wtg1%$MwXX2$^L4C@rGS1=?BPu=zoqd$ac4z_&0l!C^!LWh9UY3U`oO>F@WQ*NyB}uj{I%a#y&7;_(`gkGe'
            'L)E<L8XK@9xHMcXOK#FCc6NY6*+QH?78dEqL*LcoQtAc;f-(ie&U%(A@dwiZOdfzPDb%+^QGsdUUdbRC{Nr<IMx#(^UVfd+f-?n?'
            'i6_&*42`Svn*UCq9>}>tnETCI;w}#}?4Z*ZtV9vUJ4aQKPZuF3A?LeG=;&ry+LV9lh<SON1NyqY$aw)OR@wA(j-#dG^yKJQHj1ii'
            'x{m0n%U}wTm|`=m=}K|3K}6nXY`tT@F<BCIKBq{C)K@jN3`a833Sqm7^}WsMt3t+RWVs}E9C!AouCca~$ramN>3B5S9%(mXn`BE}'
            '^hiT&Bd^$;{{QV=TX);WwSMQXK#&KK2}#B6O4>?gyO@fuwPH(NlAYF7wMdYFDIQ+H0}!%lSO2~HHiH?=fTCoj?ddvsXiaD^nEUL@'
            'x4(^|E(ZTn%xss&xtWE{KmNXV)&1&^*R8+j-%6-_%L<~!wn?omkt0=I6hRJ4R=1cpBp0q&Tdh3aQbfD7?V`F1>N^b+3v*Oc7#T_V'
            '#~r4q=Qr=d>QGlcZQc_1wopNjsAR+SLc&^Q<1kx(%>|nlo1qp%Z`0XCr%!Y&B)d{2MyqLo(Cdkjf6-@0*-^&|nIz~0if0+84bhn#'
            'Ll?WVk5@EbC()dV%ar{be^5m+?+F!AgdDS|IMGt1;N6iufFqR$ne%=Typz4rMlce~w$7W2GzI+<lPXDI7dPkh?DF(vKznrLLo|(a'
            'W<OCWF^UIsJqzpF<^u@hAU<<#S<<2YnnWE50v8zRr_KsSkV-Bpjs@e~buuddla5vHVCo)xgV^3EOFV^)<lW#o>J6NQEx}Z9=wbm$'
            '2~DGV8u<W&!5L#3K)gOzITFWVV~Q2X8Ho)3i-du(Dsdty<NCCdj~*!utR|kLh%Iy>^KoeAx0c=!xSKmCnA|c0Y$oGsCnQw%cGh8O'
            '%ORfDKdYD&&P?^aNZxNSJd~Z*oet+nyGb)>g9;{NE+Jt-X%ldGoZLjnFO_kc!CTJXw+%=brzG4s=50<Q?_Lg)GM;9The{9+H&M(3'
            'gu2-xS=xG++V@KLQomc)UqR8O(`Gq`J=x?RxnjD*_6>#jj$a$6F{Yz6PMMl}1@BfY>`^o%xGCIha*0Fp5JHvtmE62H9<8Gq2Sb!D'
            '(3?qgEVtt{vnW-4mDD80stvq))4%*_cya#r&EWEidUNe_LRBNJs75>g)y`B?CElNeUQsR;R-c}gL?2rBQ}aLvzVehfPKg~6fA2%Y'
            'zps|b7Iwa25d)bFNf4;sud+LkSLdI)Zq2PwaINbsc|ZGzYsPc@t5t(y8-j~NlFNK0E6Yp-#8A%9HCZjitwqY5IJO=Nu$G#-h}BZL'
            '`UJ_qKIz_l@L$_7T!#ggQTdkkNW1DCMsV<FpK7hLnWr(#_=ZWwuEPG?lhgC?+P5v%NrJEoPRM=~_?fsv>vG4{ZT|LO=piitqh{DW'
            '3F5yuL+3C20D%hdlYZRYxwTdv-T4sMhU)^fVP$sT>DZ{c<c&i6({zvx9FG*#NW|zJrz_B%;=)X{u9lC8a>loOxvr>zjYp7jD5H*i'
            'PO)F9-=M-juxm!QpqHTwc`s{jJQn2bkGH_}->N{Vyx4EKE*=K{u4H`Dba!8I<h7e{%{XJPtErUiGRB$fDd~_UejW5T>2fwfuQFz>'
            '>U_vB$ZdrdS-$J@BD*CxI7Z?}`3<GRMIq@^(x#W-5lIMAAItP0HL}3T*^Jrf`ynkB2qRA-0@zVKBHy7j2qFiI3ff77wz#U_rpkJ%'
            'sL_^dC01w|f*6<Smj0;4j4uDilO1J@l3GpP@8ST6d9SIzD#A(L6*Om4moF6=J79thEI!-2E%4OLFdAuh6|78i`O_*(hZ+~ae^rPI'
            '&BK+<kLDArtmopRH)<|Kz3B7;C)f%@5hZGmI%$M)(uQlew6-#G?pvO*{MtxEUJm#O$+R#*Hwa;HSP)D<K(tC$T7iyn9TEggX)9kN'
            'R5T#?HLUdoLq)Z}f}&PRx(wEXl*y`Vb;B`>H4XkV0yL^dkW2aVT}z&t@7W`>7D9)TA6T1h^(v77%^`DdIMBvYa4IP>UtbJ<J{|n!'
            'm<7~;_%7k(Mc^zW5-U6s77|g}t#se6lo{@NcmBv~R)>0yPqS|IxKZ+Q_u1pg3JO(f?!{$+`Uy(^3F}K7-$=ZXh{dO~$*64!B3m?1'
            '@SI{_)q5DQ{06mmoa{1TyFOS7^gz#>OU|8X>nHo1<(1RfwWN5geqU-t`7-m#J%$r)1S5kw4V!74^j7&8V~eqvHA;%z(5R-SA68zN'
            'lrq2+*&OB-^<I$`g67B?mfiv3Zw^MSc9}FDlhU&eqtyS>{7|Y9!G-edGYRQBL598!T2z?j5N}ypwPGk)SuuApD74B$gWrymiE_RK'
            '?A_p7xHPd_u2u!P@4tS1asG4v<&h%ysUnB4d85ka9wBNVDnLFXD+&~DXw8AK>vCK`C+SYa!xJ_N6qvo=%1NpQex-T@Z_2^MR6Qe2'
            'qe2pbjw>IbQHkTulvN4(jIu^c*&}oqGL0Bc#UVo%5q!voh}jDE_>8e|nhu<bqOs$kY>e1UY?c;fqJmGso`fw{tW*ekFtWqoN9@xN'
            '){7!n!2}Jh7*M<a1x64#f_GTg|D9G3Zq@%$vxgWi*v^eI$UROO@8Q=H!vh9f3?E^C*j3Q9JKPg$ZZxy?pJR`iqzfj!c>-&Rs>(5!'
            'Sb%n(OedS}W9%vS*i_63|5nV#KCGYLzVbL5%LD8zkF~Y@GWHga&BgP`c94CE{be5;%oEvR?8W&2V+;zUizwez!&h=iJfraHUt+XT'
            '?NU~oaVYL;{g>7gTW>sHXgp1*{(rH-<`K4$(jHQ8+W3E~g=@8ts3wKX;-4*_VMDIe8q<b#;eMHg<6mU~;9M8=ntyN!3($ybL}Dps'
            'nV)5Oa;UNnBDo?z3M5$#ku2)mgVSV!f#jcH9x%(GQQ<a;lR2IsBGenNV{YT0FZwTE_5b!CV}bC+VKVCsA|NzvC;qVMab1=E8~xCv'
            'qxEF7uDz5!oMlW^<Z9EJX;5oLoe4-Q&_z^mwJO9G5EFl1DE+y=_<$H8SAA^C8s+6hwyhD5^yqNps1c*mj~;Q2q`*hWq;~n@e$`9k'
            '6zY{i7H(N1j7oh_YoNA*hTQJubnReBHCR3UtjHz7w62ahk$d_oQlwtJ(2+Aef?x0IfW*Fo>z1(yEMC5zsf#qc49LA|?|i3`p+Rxy'
            'dtNeG7%e8{sU0zs3btCO?66*i&DB_LaxU>wnjPc004FjgNKNF7i7}8GaV`Bb_}@PbUJhOjE>D9W2S2?$eRqCQ%~TqmYqHnJ5xSba'
            '_IKLDoax(`o@ecb%NeV&t_{E6z1jxjk0xIsi{P`^^9J9#HfJW^tWemQSo<EP`SqC%TLP;KJ0T)?J(Vkc<DL%~Z~Crv_kCX}i^SA-'
            'bGy}eiy8HNjlp)P=LaAMuO?))W3F^G0c<8WWC`2kT|OJ!^Hwi2vrah6kO=_(TEN=Q4}ycTJQ|aRRCSU|yAFaW;&nLy0$OncW0#o5'
            'Z0r2iRdwq|13%%LbpK$8jp^#>*$>xBj;bGMWl6GrTU@I!I&pG4)C{yb%}XT!q8h8vZ2C8WA#_$eEPMD}Oaj_8-wca99x5fIiijyo'
            'Fq+GWp9ufAz$1J+B$}+@v*Gt{5f5_a{%)CVV4>B0OU3A0<i@5HS<S2INRXbER7%+7vUJtvLv_=Zq@mVS-tDB|3<k?_630Z{F2;R~'
            'KYA3Kq&lDmuYMeyoM8U=lhap&v&+--Gq}8e^TTgFC0?DM3|<CIObDB&lc?r&eib$_2-ba04)RqRjPQ=9ILpy7qbA{ZFijI3q{NmX'
            '&N$I1ze$EnxbmK#{*~gh`OFVhij!eJTh10y#ys7jWLHqRm&!i|{|H#DQ~}GAMUiiiN@QBTS3Yx!Gwcy(H<mzO^K(S*BbxR<0#(BI'
            'HpREsze<0m_wHQ$upU-U_mS&-5@JrH+<{s`_Knc4+zN2>BQzAfR_vDAu|jsIfWYmHH#rGmmE;bO@@bVF4{m1>trlujha#%a-UN!w'
            'ox?Mg*w|4t$ReG?qAq)xTTZS>Pa4rRA(EfTme8lv>INfsWsk(Cu}M2R(Q$5FeCkUij>8^+`6vkAVNr@v@~t%p21$IZp|g(Pg-)e*'
            'St(Dv0`8#`XbtKLjd*bRqThkbNO)hO4=fGqBsDZaF++DL>S0uwGV-ESaw<?n1{$Dg<z0I+$mEiwiljjDLC{k>zYW&1!|zf~^e^Xr'
            'nWoDE<PK1%F(cxN;-cHN4wgEf$!>>>(U0p^+v{g>E8fHJVqrCF&Pck9s;2^R1gh83cZRV;SK11XrYfA97AG!5%4N=>v&<jtaPSha'
            '(ttc1en~C$BQKSPrguanxf<XYXzGV83!(T3gRpV*^lHdfjLCL@jOE`RU@o=>E6{l_eMa<K5TMg6F9~9lV~3IUFo|_7;ApY3Q}hD#'
            'hM)VH8TfAx^akh#Kx607*rH<pHsXl5{uUaN?OA--Cu`D8>zsn)bPLDfXI=fBBo+)l8;!VggV*df%g`&&#TADP-trcrU^cGlx>XIA'
            'k~gWJ6t^x5{Q)k8h9{f0Rn=WEon`SXfYVX+OTKz1^T5j~tZ^i%EN=YLV?7bOv*zT*vg!BAZg><WL9KLYFMo!0gFMKsVZqeY(jY^p'
            'y%w#4)pM50S0C2B!8zUOpH>kb3e-C#sQSuELQ!{4yH!x+jW*d+F(a?6Ev7jJUwX{nr#EX&yP)VHp-U-dHlubhCc%#O#E~fhMB1Z<'
            '|G;KNdLN8;B{vG4&jisH6!_v3CbXPyB04D3#WI4!QkFR8fu~)c$$1l}6gLR#?ql#xu%%J_*beBR{4`s0!?`HRXv4@T{gF!(FG&lQ'
            '58TM#mnFwia;xj5O2<}78UwDj7ktMf!SBLP=LUrv-%39RZ9hZ-mn!&1UR8`PtMoQmC}p~2>lyL}ZfgtgP#-?_^njfAEa)ImL*~=e'
            'ouX-kqMoh<!Jox=K?uOZGA)|iSR!pd>wbssrF279AE}LL06CKjWtPDZ<`u`u25|vWFbdDz;3R>qi4r%`!%}tA*{!e^QrZS}`udX3'
            'R{QFV>@ykJF{d5GY{>2+cE>P9+Em_p^iMgR>~XcW3B`P4^=KNQ+|bYPA2>7}`g#kb!ynWfV(Mj`yK|^VBi}KdNREtntaJg2A5ACp'
            '=ezFje&q%`pQ!pU##<bH(IiWAjKM-y5V9;Oo=X*Nr#|H72u9NRG3q4TCd?NuJUm34$2PejBGgqhhULi>>h_LM-oo$7%Ain5dp=bq'
            '7X6*|RdWi;X{JqpsFgh?Ra$RR2+-?)=ow>cORU^1O^+B-lL#h4O$#lX1TO-r6X&{0VVJ?rX-bVqEV%6n-}mw;=q#_yx~|ni5j(|j'
            'yfynqUtgd(m6u}8+W77F?%McsmL&1EO)EohJ0Z|!l3>F$(&UiP4g!3QZ*-93DlUoA<qgGSh&ulDJu_MDK{@OcXAmnZf(>Ef?+}Pd'
            'eX=TCx(h)~zr1o4PEY_&XEAZs*G|3Bv8!;2!bS#Hq*@8oewWotNIl4H?!4QoufJ>(j+<AOYEF=if<Dj*o*_gI1kgkNA`Oo4D&Xbj'
            'sn=cp>c4B}iJk(B6-9G1fr34+YCrr+B_dzusN6kjC_pWy0YnU<ZlY+{(oB2vT$%4qj4{3AzMD(pTDFcpRfq8P2yV=#+?=vS7^^6M'
            'Gq=sJzxQ03ueqGE-vkV1Mw;>tXThdZmd_JPter$zM>j!Vm}<*NQ{+mr?+jKiq(rObxo4Vut*9cfFpRFqHN$4)K3p)<EV{+JUF^uu'
            'TSpWep3Mjo4%_nl`E%1mR-cmyEtFANqvq2dn)Nw^>(DIB?6rUHSLZ+M-6n;)MQdQY<=&}kp9F1hcF`JEP5BPm3!L$Cot{WOGaF!k'
            '@%C)kKYMff=ZpT&r*Ga3F9w%`{>7hOs8Ja)gFXdqAPGARxQaJt47(0%XPt=i>l8W!U1T-u8N0=70J^xJ%wm+zH9|4KP8B-R7D`5-'
            '3qAaX(-H<z-Y5y<P<QPJww}EE3lQ-7DD>ezfci((nH&Y<BAzB_t)zNztov`yFWw2`ncx?6Bogc+3R^+JBaXHQxgg+V)K53Lf$bMW'
            '#mJnx5geQjXu~U8s%7igFl~>zGHCktMZIZLxt=`f-%6{e+3wI=sx)>s!@9M_7#Iw{%JOInx8kaW<Ugxqn<L?I79oIX83p$xz70Xg'
            '7O)dS*W1C(Y%-o@3~DbS?cFMq^OphIy~A2XxIalye^_0cP^kLIlWmW9#Vf@E5^^@p>T_cC`EKvLzIz?FZ1;&Kvw48C;WTAjY$i$W'
            'B=`N4W~fcaMTW_Vau6dx)E^hyV$M4%*c5Y2HHtHvNAs4G)mQdy&)-p<i*RSww?Fixnt$Ua6Lijf+v|O(yq3`o!3lVja7V%5R{P)k'
            '%>BB%e^<%;K#fs8+osv^fAxmqVKKMSQ}`Sn&H6w&K>SVzCr2G02hC?~we&l?SGUI@@nm88>Jxj|Nex_sfAf&(v|%_4v<Y?ZRCA7v'
            ')k@V-rYNx%!*!r`zYQ(cl7t$5ktFDaQYrMo(tCBVbQAV!-Ku+lUn__5$>418?sD)lh)fwU0@1HHcJab*5v8|85ejA+vBfGCZM%9P'
            '6G9k{<6_BaR2W#iOg`z0Hcz&2Dr{#fq~H7G9LV)pTyx?zJk!MERs`Vvp`A9Yvvz^)z<F9%Su{l-e&`@K6}r=fZ6Y(dLTJ{AOE%c@'
            'UcjP_KfH2qpBCu6ELIhpE336E=-mfa0{d!#YK!*WA-{Zf-{)D9bTADxGsrR;Dwq~g7OT9zziT~tc3jGrhG>UZh#!5s26Z|95y8+&'
            '+`K4On1+g^DY-Qe3@_UOVe#2!SYl5p85g`p|6e|5RTYh7I-{Hh+svcwDw=a%$B;HonG^>!h>PTXK}60ek`~6qaXv}c+l*M-;(+SM'
            'zfyEBj9)<G2eKHFMB_tvJ!6eev?gSgTyH3ZTGX7E$nqRKf4*ENQ=~AB(D2dua5Kx1aP2U7y}iWrlqY2Uh2M+WI7@Sk`Nvc%C0!ja'
            'tW~tw=7bi3{ddf0k^0=1ZOZcnWtVHjf#7!z37OZ>_E=$m)X52S;+|=WJ!d?oSLdqELq$^d&w0c4j_yjc$-2-{@Y$HzyP%lBRsjh('
            'm`9r>3`g~7e)smLmxD9RK}9s_-X!N3vKqWA<``)|5Af^;SuV$XFwr)PrjGoL0{bKaxgGJ_(<Dpgi&=0J%@?4MfmC@jTL<YPE~a=H'
            'Y?3Xg!pX)u^2H*fmonVDtJ!MWq$LS!^~gK}3@PC4yZ*)L(avO~gbL~gFQO$c^Fv(Ff}+)9WdyD8u+3ID!HC|S5$NW=!TPZP)ddQ}'
            '*zjvOa7L}9*X$gb82w4mD)_A<V*UwmN9GI'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
youtube_data = _local_pipeline_modules['youtube_data']
viral_prediction = _local_pipeline_modules['viral_prediction']
google_ai = _local_pipeline_modules['google_ai']



In [ ]:
# Notebook 04 ile ayni kalici yol cozumleme (Faz 9): Drive (Colab) / /content (Colab, Drive kapali) / yerel results/
youtube_paths = youtube_data.resolve_project_paths(
    is_colab=IS_COLAB,
    use_google_drive=USE_GOOGLE_DRIVE,
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
)
print("YouTube veri arama dizini (Notebook 04 ciktisi bekleniyor):", youtube_paths["youtube_data_dir"])


## 1. Gerçek geçmiş veri setini yükle (Notebook 04 çıktısı)

Notebook 04'ün yazdığı `normalized_videos.csv` + `collection_manifest.json` dosyaları önce Drive-kalıcı
konumda, sonra Colab geçici `/content/results/...` konumunda, sonra yerel proje `results/...` konumunda
aranır (Faz 9). `collection_manifest.json` okunur ve toplama durumu `failed`/`insufficient_data` ise bu
notebook **kendinden emin bir tahmin üretmeyi reddeder** — sadece ham geçmiş satır sayısına değil, Notebook
04'ün kendi topladığı verinin güvenilirliğine de bakılır.

**Önemli sınır:** Burada kullanılan `view_count`/`like_count`/`comment_count` **herkese açık** YouTube Data
API v3 istatistikleridir. Bunlar YouTube Analytics'in **özel** metrikleriyle (CTR, izlenme süresi/retention,
gelir) **aynı şey değildir** — bu notebook hiçbir zaman CTR/retention/gelir tahmini yapmaz, sadece
görüntülenme/beğeni/yorum sayıları üzerinden basit istatistiksel bir taban çizgisi kurar.

In [ ]:
def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None

NORMALIZED_VIDEOS_PATH = ''  # elle yol vermek istersen buraya yaz

# Faz 9: Drive-kalici -> Colab gecici /content -> yerel results/ sirasiyla ara.
youtube_data_search_dirs = [
    str(youtube_paths["youtube_data_dir"]),
    str(RESULTS_DIR / 'youtube_data'),
    'results/youtube_data',
    '../results/youtube_data',
    '/content/results/youtube_data',
]
normalized_path = None
if NORMALIZED_VIDEOS_PATH and Path(NORMALIZED_VIDEOS_PATH).is_file():
    normalized_path = Path(NORMALIZED_VIDEOS_PATH).resolve()
else:
    found = youtube_data.find_normalized_videos_csv(youtube_data_search_dirs)
    if found is not None:
        normalized_path = found

# collection_manifest.json: Notebook 04'un TOPLAMA SURECININ KENDISI hakkindaki
# beyani. Satir sayisi yeterli olsa bile, manifest status'u 'failed' veya
# 'insufficient_data' ise bu notebook kendinden emin bir tahmin URETMEZ (asagidaki
# veri yeterliligi kapisinde kontrol edilir).
collection_manifest = None
if normalized_path is not None:
    manifest_path = normalized_path.parent / 'collection_manifest.json'
    if manifest_path.is_file():
        try:
            with open(manifest_path, 'r', encoding='utf-8') as f:
                collection_manifest = json.load(f)
        except Exception as exc:
            print(f'UYARI: collection_manifest.json okunamadi: {exc}')

collection_status = collection_manifest['status'] if collection_manifest else None
print('Notebook 04 toplama durumu (collection_manifest.json):', collection_status or '(manifest bulunamadi)')

# Faz 9: onbellek/veri bayatligi uyarisi -- toplama ne kadar eski?
YOUTUBE_DATA_STALENESS_WARNING_HOURS = 48
if collection_manifest and collection_manifest.get('completed_at'):
    try:
        completed_at = datetime.fromisoformat(collection_manifest['completed_at'])
        if completed_at.tzinfo is None:
            completed_at = completed_at.replace(tzinfo=timezone.utc)
        age_hours = (datetime.now(timezone.utc) - completed_at).total_seconds() / 3600.0
        if age_hours > YOUTUBE_DATA_STALENESS_WARNING_HOURS:
            print(f'UYARI: YouTube verisi {age_hours:.1f} saat once toplanmis '
                  f'(esik: {YOUTUBE_DATA_STALENESS_WARNING_HOURS}s). Guncel sonuclar icin Notebook 04\'u tekrar calistir.')
    except Exception:
        pass

# Faz 6 duzeltmesi: Notebook 04 artik normalized_videos.csv'yi HER ZAMAN sabit
# semali yazar, ama bu notebook yine de kendini savunmali okumali - dosya elle
# silinmis/bozulmus/baska bir surumden kalma olabilir. Hicbir durumda cokmez:
# headerless/tamamen bos CSV -> EmptyDataError yakalanir; gerekli sutunlar eksikse
# ayri bir kontrolle yakalanir. Her iki durumda da history_df sabit semali BOS
# bir DataFrame'e duser ve asagidaki veri yeterliligi kapisi bunu insufficient_data
# olarak raporlar.
EMPTY_HISTORY_COLUMNS = [
    'video_id', 'title', 'channel_id', 'channel_title', 'published_at',
    'view_count', 'like_count', 'comment_count', 'duration_iso8601',
]
REQUIRED_HISTORY_COLUMNS = ['video_id', 'channel_id', 'view_count', 'published_at']

history_df = pd.DataFrame(columns=EMPTY_HISTORY_COLUMNS)
history_load_status = 'insufficient_data'
history_load_reason = 'normalized_videos.csv bulunamadi'

if normalized_path is None:
    print(f'UYARI: {history_load_reason}. Once Notebook 04\'u YOUTUBE_DATA_API_KEY ve '
          'gercek SEED_CHANNEL_IDS ile calistir. history_df bos kalacak.')
else:
    raw_history_df = None
    try:
        raw_history_df = pd.read_csv(normalized_path)
    except pd.errors.EmptyDataError:
        history_load_reason = f'{normalized_path} tamamen bos (basliksiz/0 byte) bir CSV dosyasi'
        print(f'UYARI: {history_load_reason}. history_df bos kalacak, tahmin uretilmeyecek.')
    except Exception as exc:
        history_load_reason = f'{normalized_path} okunurken hata: {exc}'
        print(f'UYARI: {history_load_reason}. history_df bos kalacak.')

    if raw_history_df is not None:
        missing_columns = [c for c in REQUIRED_HISTORY_COLUMNS if c not in raw_history_df.columns]
        if missing_columns:
            history_load_reason = (
                f'gerekli sutunlar eksik: {missing_columns} '
                f'(dosyada bulunanlar: {list(raw_history_df.columns)}). Dosya bozuk veya eski/uyumsuz '
                'bir Notebook 04 semasindan kalma olabilir.'
            )
            print(f'UYARI: {history_load_reason}. history_df bos kalacak.')
        else:
            candidate_df = raw_history_df.dropna(subset=['view_count']).copy()
            if 'published_at' in candidate_df.columns:
                candidate_df['published_at'] = pd.to_datetime(candidate_df['published_at'], errors='coerce', utc=True)
                candidate_df = candidate_df.sort_values('published_at')
            if len(candidate_df) == 0:
                history_load_reason = (
                    f'{normalized_path} gecerli semaya sahip ama kullanilabilir (view_count dolu) satir yok'
                )
                print(f'UYARI: {history_load_reason}. history_df bos kalacak.')
            else:
                history_df = candidate_df
                history_load_status = 'ok'
                history_load_reason = ''
                print('Yuklendi:', normalized_path, f'({len(history_df)} kullanilabilir satir, view_count dolu)')

## 2. Manuel örnekler — SADECE deneysel fallback

Bu örnekler asla gerçek geçmiş veri setine karıştırılmaz ve asla `production_ready`/güvenilir tahmin üretmek için kullanılmaz — sadece heuristik formülün mantığını göstermek için elle doldurulabilir.

In [ ]:
MANUAL_EXAMPLES = [
    {
        'title': 'Bu Sahneyi Kimse Beklemiyordu!',
        'duration_sec': 58,
        'channel_avg_views_last_10': 42000,
        'channel_avg_ctr': 0.061,
    },
]

def heuristic_predict(v):
    base = v['channel_avg_views_last_10']
    duration_penalty = 1.0 if 30 <= v['duration_sec'] <= 90 else 0.85
    ctr_factor = v['channel_avg_ctr'] / 0.05
    return base * duration_penalty * ctr_factor

manual_preds = [round(heuristic_predict(v)) for v in MANUAL_EXAMPLES]
log_result(
    'viral-prediction', 'manual-example-heuristic', None, 0.0, 0.0,
    f'SADECE gosterim amacli, {len(MANUAL_EXAMPLES)} elle girilen ornek: {manual_preds}',
    status='experimental',
)
print('Manuel ornek tahminleri (SADECE gosterim, production icin kullanilmaz):', manual_preds)

## 3. Veri yeterliliği kapısı

In [ ]:
MIN_HISTORY_ROWS = 20  # bu esigin altinda notebook kendinden emin tahmin URETMEZ

HISTORY_ROW_COUNT = len(history_df)
COLLECTION_STATUS_OK = collection_status not in ('failed', 'insufficient_data')
DATA_SUFFICIENT = (HISTORY_ROW_COUNT >= MIN_HISTORY_ROWS) and COLLECTION_STATUS_OK

date_range = {'start': None, 'end': None}
if HISTORY_ROW_COUNT and 'published_at' in history_df.columns and history_df['published_at'].notna().any():
    date_range['start'] = history_df['published_at'].min().isoformat()
    date_range['end'] = history_df['published_at'].max().isoformat()

print(f'Gecmis satir sayisi: {HISTORY_ROW_COUNT} (esik: {MIN_HISTORY_ROWS})')
print('Veri yukleme durumu:', history_load_status, '-', history_load_reason or '(sorun yok)')
print('Notebook 04 toplama durumu:', collection_status or '(manifest bulunamadi)')
print('Veri tarih araligi:', date_range)
if not COLLECTION_STATUS_OK:
    print(f"UYARI: collection_manifest.json status='{collection_status}' -- Notebook 04'un toplama sureci "
          "basarisiz/yetersizdi. Satir sayisi yeterli olsa bile kendinden emin tahmin URETILMEYECEK.")
elif not DATA_SUFFICIENT:
    print(f'UYARI: Yetersiz veri (status=insufficient_data olacak). En az {MIN_HISTORY_ROWS} satir icin '
          "Notebook 04'u daha fazla YOUTUBE_CHANNEL_INPUTS ile calistir.")
else:
    print('Veri yeterli; baseline degerlendirmesi calisacak.')


## 4. Şeffaf baseline yöntemler

Üç açıklanabilir, ücretsiz, API gerektirmeyen baseline: kanal medyan izlenme, son N videonun hareketli medyan/ortalaması, etkileşim (like/comment oranı) ile ayarlanmış medyan.

In [ ]:
# Kanonik baseline yontemleri (local_pipeline/viral_prediction.py). Master
# notebook da AYNI fonksiyonlari cagirir -- iki ayri kopya yoktur.
BASELINE_METHODS = viral_prediction.BASELINE_METHODS
ROLLING_WINDOW = viral_prediction.DEFAULT_ROLLING_WINDOW
zero_safe_smape = viral_prediction.zero_safe_smape
print(f'{len(BASELINE_METHODS)} kanonik baseline yontemi yuklendi:', list(BASELINE_METHODS))


## 5. Zaman-tabanlı holdout değerlendirmesi

Veri yayın tarihine göre sıralanır; son `%HOLDOUT_FRACTION` bölüm test (holdout), geri kalanı eğitim/referans olarak kullanılır — bu, gerçek üretim senaryosuna (geçmişten geleceği tahmin) daha yakın bir değerlendirmedir.

In [ ]:
HOLDOUT_FRACTION = 0.2
MIN_HOLDOUT_ROWS = 5

baseline_metrics = {}
train_df, holdout_df = pd.DataFrame(), pd.DataFrame()

if DATA_SUFFICIENT:
    t0 = time.time()
    # Kanonik zaman-tabanli holdout degerlendirmesi (master notebook ile ayni).
    evaluation = viral_prediction.evaluate_baselines_on_holdout(
        history_df, holdout_fraction=HOLDOUT_FRACTION, min_holdout_rows=MIN_HOLDOUT_ROWS,
    )
    train_df = evaluation['train_df']
    holdout_df = evaluation['holdout_df']
    baseline_metrics = evaluation['metrics']
    latency = time.time() - t0

    if len(holdout_df) < MIN_HOLDOUT_ROWS:
        print(f'UYARI: holdout seti cok kucuk ({len(holdout_df)} satir); metrikler guvenilmez olabilir.')

    for method_name in BASELINE_METHODS:
        if method_name in baseline_metrics:
            m = baseline_metrics[method_name]
            log_result('viral-prediction', method_name, None, 0.0, latency / max(len(BASELINE_METHODS), 1),
                       f"MAE={m['mae']:.0f} medianAE={m['median_ae']:.0f} sMAPE={m['smape_pct']}% n={m['sample_count']}",
                       status='review_required')
        else:
            log_result('viral-prediction', method_name, None, 0.0, 0.0,
                       'holdout icin tahmin uretilemedi', status='insufficient_data')
    print(json.dumps(baseline_metrics, indent=2))
else:
    log_result('viral-prediction', 'baseline-evaluation', None, 0.0, 0.0,
                f'yetersiz veri ({HISTORY_ROW_COUNT} < {MIN_HISTORY_ROWS})', status='insufficient_data')
    print('Baseline degerlendirmesi atlandi: yetersiz veri.')


## 6. Opsiyonel LLM tahmini (istatistiksel baseline ile karşılaştırılır)

**Kalite politikası:** LLM tahmini SADECE `DATA_SUFFICIENT=True` iken çalışır ve SADECE gerçek geçmiş holdout verisiyle beslenir — asla `MANUAL_EXAMPLES`'tan güven uydurmaz. Küçük bir holdout örneklemiyle sınırlıdır (maliyet + anlamlılık için).

In [ ]:
LLM_HOLDOUT_SAMPLE_SIZE = 5
llm_metrics = None
llm_provider_used = None

# Faz 8 duzeltmesi: OpenAI response_format={'type':'json_object'} bir JSON NESNESI
# ister, dizi degil. Onceki surum bunu isterken promptta hala bir JSON DIZISI
# istiyordu - tutarsizdi. Simdi TUM saglayicilar icin ayni sema istenir: ust
# seviyede {"predictions": [...]}. Her tahmin, listedeki POZISYONA gore degil,
# video_id (sabit/kararli bir tanimlayici) ile eslestirilir.
def parse_prediction_response(raw):
    try:
        start = raw.find('{'); end = raw.rfind('}') + 1
        if start == -1 or end <= start:
            return None
        obj = json.loads(raw[start:end])
        if not isinstance(obj, dict) or not isinstance(obj.get('predictions'), list):
            return None
        return obj['predictions']
    except Exception:
        return None

if not DATA_SUFFICIENT:
    print('LLM tahmini atlandi: yetersiz gercek veri (birkac elle girilen ornekten guven uydurulmaz).')
elif not (OPENAI_API_KEY or ANTHROPIC_API_KEY or GOOGLE_API_KEY):
    print('LLM tahmini atlandi: API anahtari yok.')
else:
    llm_sample = holdout_df.head(LLM_HOLDOUT_SAMPLE_SIZE)
    channel_context = (
        train_df.groupby('channel_id')['view_count']
        .median().round(0).to_dict()
    )
    prompt_payload = [
        {
            'video_id': str(row['video_id']),
            'title': row['title'],
            'channel_id': row['channel_id'],
            'channel_median_views': channel_context.get(row['channel_id']),
            'published_at': str(row['published_at']),
        }
        for _, row in llm_sample.iterrows()
    ]
    PREDICTION_PROMPT = (
        "Sen bir YouTube analitik uzmanisin. Asagidaki gercek videolarin her biri icin, verilen kanal "
        "medyan izlenme sayisina gore beklenen izlenme sayisini (predicted_views, tek sayi) ve 0-1 arasi "
        "guven skoru (confidence) ver. Her tahmin icin verilen video_id degerini AYNEN geri don - "
        "eslestirme icin kullanilacak, liste sirasina guvenilmeyecek. TUM videolar icin tahmin uret, "
        "hicbirini atlama. SADECE su sema ile bir JSON NESNESI don (dizi DEGIL): "
        "{{\"predictions\": [{{\"video_id\": \"...\", \"predicted_views\": <int>, \"confidence\": <0-1 sayi>}}]}}\n\n"
        "Veri:\n{data}"
    ).format(data=json.dumps(prompt_payload, ensure_ascii=False))

    predictions, provider_name, cost, latency = None, None, 0.0, 0.0
    try:
        if OPENAI_API_KEY:
            from openai import OpenAI
            client = OpenAI(api_key=OPENAI_API_KEY)
            t0 = time.time()
            resp = client.chat.completions.create(
                model='gpt-5.6-luna', messages=[{'role': 'user', 'content': PREDICTION_PROMPT}],
                temperature=0, response_format={'type': 'json_object'},
            )
            latency = time.time() - t0
            raw_text = resp.choices[0].message.content
            predictions = parse_prediction_response(raw_text)
            provider_name = 'openai-gpt-5.6-luna'
            usage = resp.usage
            cost = (usage.prompt_tokens / 1_000_000) * 1.00 + (usage.completion_tokens / 1_000_000) * 6.00
        elif ANTHROPIC_API_KEY:
            import anthropic
            client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            t0 = time.time()
            resp = client.messages.create(
                model='claude-haiku-4-5', max_tokens=600,
                messages=[{'role': 'user', 'content': PREDICTION_PROMPT}],
            )
            latency = time.time() - t0
            predictions = parse_prediction_response(resp.content[0].text)
            provider_name = 'anthropic-claude-haiku-4-5'
            cost = (resp.usage.input_tokens / 1_000_000) * 1.00 + (resp.usage.output_tokens / 1_000_000) * 5.00
        elif GOOGLE_API_KEY:
            # Google: SABIT MODEL YOK -- kanonik router (local_pipeline/google_ai.py).
            # Viral sonucun yorumlanmasi KRITIK OLMAYAN bir istir: Rota B
            # (gemini-3.5-flash-lite -> gemini-3.1-flash-lite -> gemma-4-26b-a4b-it).
            # Istatistiksel baseline'in YERINE GECMEZ; onunla karsilastirilir.
            _g_credential = google_ai.resolve_secret('google_api', get_secret)
            _g_client = google_ai.build_google_client(_g_credential)
            _g_manifest = google_ai.build_google_capability_manifest(
                _g_client, credential=_g_credential)
            _g_ledger = google_ai.QuotaLedger()
            print(google_ai.format_route_line(
                google_ai.select_google_route('viral_interpretation',
                                              manifest=_g_manifest, ledger=_g_ledger),
                _g_credential))
            _g_result = google_ai.run_google_viral_interpretation(
                PREDICTION_PROMPT, client=_g_client, manifest=_g_manifest, ledger=_g_ledger)
            latency = _g_result.get('latency_sec', 0.0)
            provider_name = 'google-' + str(_g_result.get('model') or 'router')
            cost = 0.0  # ucretsiz katman; gercek butce RPM/TPM/RPD kotasidir
            if _g_result['status'] == 'ok':
                predictions = parse_prediction_response(_g_result['output'])
            else:
                # Cikti UYDURULMAZ: predictions None kalir -> asagida 'failed' loglanir.
                predictions = None
                print('Google rotasi basarisiz:', _g_result['status_reason'])

        if predictions is None:
            log_result('viral-prediction', provider_name or 'llm-unknown', None, cost, latency,
                        "LLM cevabi ayristirilamadi (beklenen sema: {'predictions': [...]})", status='failed')
        else:
            # video_id ile eslestir (liste POZISYONUNA guvenme - Faz 8 duzeltmesi).
            # Gercek holdout view_count degerleri (actuals) HICBIR sekilde LLM
            # ciktisindan degil, dogrudan llm_sample'dan alinir.
            llm_sample_by_id = {str(row['video_id']): row for _, row in llm_sample.iterrows()}
            matched_ids, matched_preds, matched_actuals = [], [], []
            unknown_ids, malformed_count = [], 0
            for entry in predictions:
                if not isinstance(entry, dict) or 'video_id' not in entry or 'predicted_views' not in entry:
                    malformed_count += 1
                    continue
                vid = str(entry['video_id'])
                if vid not in llm_sample_by_id:
                    unknown_ids.append(vid)
                    continue
                if vid in matched_ids:
                    continue  # duplicate video_id in response - ignore repeats
                matched_ids.append(vid)
                matched_preds.append(entry['predicted_views'])
                matched_actuals.append(float(llm_sample_by_id[vid]['view_count']))

            expected_count = len(llm_sample)
            matched_count = len(matched_ids)
            is_complete = (matched_count == expected_count) and malformed_count == 0 and not unknown_ids
            completeness_note = (
                'tam' if is_complete else
                f'EKSIK/BOZUK ({matched_count}/{expected_count} eslesti, malformed={malformed_count}, bilinmeyen_video_id={len(unknown_ids)})'
            )

            if not matched_preds:
                log_result('viral-prediction', provider_name, None, cost, latency,
                            f'hic gecerli eslesen tahmin yok ({completeness_note})', status='failed')
            elif not is_complete:
                # Faz 8 duzeltmesi: eksik/bozuk cikti acikca 'failed' olarak isaretlenir,
                # kismi sonuc sessizce baseline ile karsilastirmaya sokulmaz.
                log_result('viral-prediction', provider_name, None, cost, latency,
                            f'eksik/bozuk LLM ciktisi, sonuc guvenilir sayilmadi ({completeness_note})', status='failed')
            else:
                mae = float(np.mean(np.abs(np.array(matched_preds) - np.array(matched_actuals))))
                median_ae = float(np.median(np.abs(np.array(matched_preds) - np.array(matched_actuals))))
                smape = zero_safe_smape(matched_actuals, matched_preds)
                llm_metrics = {
                    'mae': round(mae, 1), 'median_ae': round(median_ae, 1),
                    'smape_pct': round(smape, 2) if smape is not None else None,
                    'sample_count': matched_count, 'expected_count': expected_count, 'complete': True,
                }
                llm_provider_used = provider_name
                log_result('viral-prediction', provider_name, None, cost, latency,
                            f"MAE={mae:.0f} medianAE={median_ae:.0f} n={matched_count} ({completeness_note}, baseline ile karsilastir)",
                            status='review_required')
    except Exception as exc:
        log_result('viral-prediction', provider_name or 'llm-unknown', None, cost, latency, f'API hatasi: {exc}', status='failed')

    if llm_metrics and baseline_metrics:
        best_baseline_mae = min(m['mae'] for m in baseline_metrics.values())
        llm_mae = llm_metrics['mae']
        comparison_word = 'daha iyi' if llm_mae < best_baseline_mae else 'daha kotu ya da esit'
        print(f'LLM MAE={llm_mae} vs en iyi baseline MAE={best_baseline_mae} ({llm_provider_used} {comparison_word})')

## 7. Dışa aktarım

Tek bir "kesin" sayı yerine **aralık** (`predicted_range`) ve açık güven seviyesi/sınırlamalar raporlanır.

In [ ]:
df = pd.DataFrame(results)
out_path = RESULTS_DIR / '05_viral_prediction_sonuclar.csv'
if not df.empty:
    display(df)
    df.to_csv(out_path, index=False)
else:
    pd.DataFrame(columns=['task', 'candidate', 'quality_score_1_5', 'cost_usd_estimate', 'latency_sec', 'notes', 'status']).to_csv(out_path, index=False)
print('Kaydedildi:', out_path)

limitations = []
if history_load_status != 'ok':
    limitations.append(f'gecmis veri yuklenemedi/yetersiz: {history_load_reason}')
if not DATA_SUFFICIENT:
    limitations.append(f'gercek gecmis veri yetersiz ({HISTORY_ROW_COUNT} < {MIN_HISTORY_ROWS} satir)')
if HISTORY_ROW_COUNT and len(holdout_df) < MIN_HOLDOUT_ROWS:
    limitations.append(f'holdout seti kucuk ({len(holdout_df)} satir); metrikler gurultulu olabilir')
if llm_metrics is None:
    limitations.append('LLM tahmini calismadi, atlandi veya eksik/bozuk cikti verdi; sadece istatistiksel baseline mevcut')
limitations.append('YouTube Shorts algoritmasindaki degisiklikler, mevsimsellik ve konu turu farkliliklari modele dahil edilmemistir')

if not DATA_SUFFICIENT:
    prediction_status = 'insufficient_data'
    status_reason = f'gercek gecmis veri {MIN_HISTORY_ROWS} satir esiginin altinda ({HISTORY_ROW_COUNT} satir) - {history_load_reason or "yeterli veri toplanamadi"}'
    predicted_range = None
    confidence_level = 'none'
    chosen_method = None
elif baseline_metrics:
    chosen_method = min(baseline_metrics, key=lambda name: baseline_metrics[name]['mae'])
    chosen = baseline_metrics[chosen_method]
    center = float(train_df['view_count'].median())
    predicted_range = {'low': round(max(center - chosen['mae'], 0), 0), 'high': round(center + chosen['mae'], 0)}
    if HISTORY_ROW_COUNT >= 200:
        confidence_level = 'medium'
    elif HISTORY_ROW_COUNT >= 50:
        confidence_level = 'low'
    else:
        confidence_level = 'very_low'
    prediction_status = 'review_required'
    status_reason = f'istatistiksel baseline ({chosen_method}) holdout uzerinde degerlendirildi; insan onayi bekleniyor'
else:
    chosen_method = None
    predicted_range = None
    confidence_level = 'none'
    prediction_status = 'failed'
    status_reason = 'veri yeterliydi ama hicbir baseline holdout tahmini uretemedi'

viral_prediction = {
    'status': prediction_status,
    'status_reason': status_reason,
    'dataset_size': HISTORY_ROW_COUNT,
    'date_range': date_range,
    'min_history_rows_threshold': MIN_HISTORY_ROWS,
    'history_load_status': history_load_status,
    'history_load_reason': history_load_reason or None,
    'method': chosen_method,
    'llm_provider_model': llm_provider_used,
    'metrics': {
        'baseline': baseline_metrics,
        'llm': llm_metrics,
    },
    'predicted_range': predicted_range,
    'confidence_level': confidence_level,
    'limitations': limitations,
    'generated_at': datetime.now(timezone.utc).isoformat(),
}

viral_prediction_path = RESULTS_DIR / 'viral_prediction.json'
with open(viral_prediction_path, 'w', encoding='utf-8') as f:
    json.dump(viral_prediction, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', viral_prediction_path, '(status:', prediction_status, ')')

zip_path = RESULTS_DIR / '05_viral_prediction_paylasim_paketi.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(viral_prediction_path, arcname='viral_prediction.json')
    if out_path.exists():
        archive.write(out_path, arcname=out_path.name)
print('Paylasim paketi hazir:', zip_path)

## 8. Karar (elle doldurulacak)

Not: Gerçek üretim verisi biriktikçe (Faz 9 — Öğrenme Döngüsü) bu karar `ai_call_logs` + `analytics_snapshots` verisiyle yeniden değerlendirilecek. `MIN_HISTORY_ROWS` eşiği altında hiçbir yöntem `production_ready` sayılmamalıdır.

- **Seçilen yaklaşım:** _...
- **Gerekçe:** _...
